In [ ]:
# FIRST CELL in a fresh Colab runtime. Restart session once after installation.
# Transformers 4.x uses the native SentencePiece tokenizer for BanglaT5.
%pip install -q "transformers==4.51.3" "tokenizers>=0.21,<0.22" "sentencepiece==0.2.1" "protobuf>=5.29.0,<8" "accelerate>=1.2,<2" "datasets>=3,<4" "sacrebleu" "rouge-score" "rapidfuzz" "openpyxl" "scikit-learn" "matplotlib" "seaborn" "joblib" "nltk" "psutil"
print('RESTART SESSION now; then run the Drive cell onward. Do not rerun another package-install cell later.')


  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio-client 2.7.2 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.31.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
gcsfs 2025.12.0 requires fsspec==2025.12.0, but you have fsspec 2025.3.0 which is incompatible.
diffusers 0.42.1 requires huggingface-hub<3.0,>=1.32.0, but you have huggingface-hub 0.36.2 which is incompatible.
RESTART SESSION now; then run the Drive cell onward. Do not rerun another package-install cell later.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


Locate and select the INTERNAL corpus (no guessed filenames)

In [ ]:

from pathlib import Path
import os, re, json, random, time, unicodedata, hashlib, gc, warnings
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns, joblib
from collections import defaultdict, Counter
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import (accuracy_score, f1_score, precision_recall_fscore_support,
    classification_report, confusion_matrix, roc_curve, auc, roc_auc_score, log_loss)
from sklearn.preprocessing import label_binarize
from scipy.special import softmax
from scipy import sparse
import torch
SEED=42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
ROOT=Path('/content/drive/MyDrive/BANGLA DIALECT PAPER/DATASET')
EXT_ROOT=ROOT/'external'/'external_raw'
OUT=ROOT/'reviewer_ready_v8_reviewer_complete'
for sub in ['audit','splits','prepared_external','models','predictions','metrics','figures','logs']:
    (OUT/sub).mkdir(parents=True,exist_ok=True)
# Set explicit raw internal path if auto-discovery finds >1 plausible file.
INTERNAL_FILE=ROOT/'Bangla_Dialect_Dataset.csv'
RUN_CLASSICAL=True
RUN_TRANSFORMERS=True
RUN_LSTM=True
RUN_GENERATORS=True
RUN_EXTERNAL_WORD_LEVEL=False
NEAR_THRESHOLD=0.90 # character TF-IDF cosine; research operational threshold, not semantic proof
NEAR_NGRAM=(3,5)
TRAIN_FRAC,VAL_FRAC,TEST_FRAC=.70,.15,.15
CLASSIFIER_EPOCHS=4
GENERATOR_EPOCHS=3
LSTM_EPOCHS=15
CLASSIFIER_BATCH=8
GENERATOR_BATCH=2
GEN_MAX_INPUT=96
GEN_MAX_TARGET=96
LABELS=['Barisal','Chittagong','Khulna','Noakhali','Sylhet']
L2I={s:i for i,s in enumerate(LABELS)}
I2L={i:s for s,i in L2I.items()}
plt.rcParams.update({'font.family':'DejaVu Sans','figure.dpi':120,'savefig.dpi':300})
print('GPU:',torch.cuda.is_available(),'Output:',OUT)


GPU: True Output: /content/drive/MyDrive/BANGLA DIALECT PAPER/DATASET/reviewer_ready_v8_reviewer_complete


Mandatory tokenizer compatibility check (run after runtime restart)

Code cell 7

In [ ]:
import google.protobuf, sentencepiece, transformers
from transformers import AutoTokenizer
print('protobuf',google.protobuf.__version__,'sentencepiece',sentencepiece.__version__,'transformers',transformers.__version__)
assert transformers.__version__.startswith('4.51.'), 'Restart runtime after the installation cell'
try:
    _bangla_tok=AutoTokenizer.from_pretrained('csebuetnlp/banglat5',use_fast=False)
    assert len(_bangla_tok('আমি বাংলায় কথা বলি।')['input_ids'])>0
    print('BanglaT5 slow tokenizer: OK')
finally:
    if '_bangla_tok' in globals(): del _bangla_tok


protobuf 6.33.6 sentencepiece 0.2.1 transformers 4.51.3


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/1.11M [00:00<?, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


BanglaT5 slow tokenizer: OK


Locate and load the original internal dataset (do not load generated outputs)


Code cell 8

In [ ]:
def read_internal(p):
    if p.suffix.lower() in ['.xlsx','.xls']: return pd.read_excel(p)
    for sep in [',','\t']:
        try:
            d=pd.read_csv(p,sep=sep,encoding='utf-8-sig')
            if {'region','dialect_sentence'}.issubset(d.columns):return d
        except Exception: pass
    raise ValueError('Cannot read expected internal columns from '+str(p))
if INTERNAL_FILE is None:
    candidates=[]
    for p in ROOT.rglob('*'):
        if not p.is_file() or p.suffix.lower() not in ('.csv','.xlsx','.xls'):continue
        if 'external' in [x.lower() for x in p.parts] or 'reviewer_ready' in str(p).lower():continue
        if 'dialect' not in p.name.lower() and 'internal' not in p.name.lower():continue
        try:
            d0=read_internal(p)
            if {'region','dialect_sentence'}.issubset(d0.columns): candidates.append(p)
        except Exception:pass
    print('Internal candidates:');[print(' ',p) for p in candidates]
    if len(candidates)!=1:raise RuntimeError('Set INTERNAL_FILE=Path("/content/drive/MyDrive/.../your_original_internal.csv") in configuration cell, then rerun')
    INTERNAL_FILE=candidates[0]
INTERNAL_FILE=Path(INTERNAL_FILE)
raw=read_internal(INTERNAL_FILE)
print('SOURCE:',INTERNAL_FILE,'rows:',len(raw),'columns:',raw.columns.tolist())
assert 'region' in raw and 'dialect_sentence' in raw
ref_candidates=['standard_bengali','standard_Bangla','standard_bangla','Standard_Bangla','standard_bengali_sentence']
ref_col=next((c for c in ref_candidates if c in raw.columns),None)
if ref_col is None:raise ValueError('Set actual standard Bangla column mapping; found '+str(raw.columns.tolist()))
raw=raw.rename(columns={ref_col:'standard_bengali'})


SOURCE: /content/drive/MyDrive/BANGLA DIALECT PAPER/DATASET/Bangla_Dialect_Dataset.csv rows: 23440 columns: ['region', 'dialect_sentence', 'standard_bengali']


3. Minimal Unicode normalization and data quality audit
   Code cell 8


Code cell 10

In [ ]:
def norm(x):
    if pd.isna(x):return ''
    s=unicodedata.normalize('NFC',str(x)).replace('\u200b','').replace('\ufeff','')
    return re.sub(r'\s+',' ',s).strip()
ALIASES={'barishal':'Barisal','barisal':'Barisal','chittagong':'Chittagong','chattogram':'Chittagong',
         'khulna':'Khulna','noakhali':'Noakhali','nowakhali':'Noakhali','sylhet':'Sylhet','sylheti':'Sylhet'}
df=raw[['region','dialect_sentence','standard_bengali']].copy()
df['source_row']=np.arange(len(df))
df['region']=df.region.map(lambda x:ALIASES.get(norm(x).lower(),norm(x)))
for c in ['dialect_sentence','standard_bengali']:df[c]=df[c].map(norm)
quality={'raw_rows':len(df)}
valid=df.region.isin(LABELS)&df.dialect_sentence.ne('')&df.standard_bengali.ne('')
quality['invalid_or_unmapped']=int((~valid).sum())
df=df[valid].copy()
# Identical rows are not independent observations.
quality['exact_triplet_duplicates']=int(df.duplicated(['region','dialect_sentence','standard_bengali']).sum())
df=df.drop_duplicates(['region','dialect_sentence','standard_bengali']).reset_index(drop=True)
conflict=df.groupby('dialect_sentence').agg(n_regions=('region','nunique'),n_refs=('standard_bengali','nunique'))
ambig_inputs=set(conflict.index[(conflict.n_regions>1)|(conflict.n_refs>1)])
quality['ambiguous_input_strings']=len(ambig_inputs)
# Exclude ambiguous training labels/references rather than silently pick one.
ambig=df[df.dialect_sentence.isin(ambig_inputs)].copy()
ambig.to_csv(OUT/'audit'/'ambiguous_internal_rows.csv',index=False)
df=df[~df.dialect_sentence.isin(ambig_inputs)].reset_index(drop=True)
# Preserve only cleaned observations. Length restriction applies to dialect input.
df['token_count']=df.dialect_sentence.str.split().str.len()
quality['outside_4_60_tokens']=int((~df.token_count.between(4,60)).sum())
df=df[df.token_count.between(4,60)].drop(columns='token_count').reset_index(drop=True)
quality['clean_rows']=len(df)
(OUT/'audit'/'quality_summary.json').write_text(json.dumps(quality,indent=2),encoding='utf-8')
df.to_csv(OUT/'audit'/'internal_cleaned_before_split.csv',index=False,encoding='utf-8-sig')
df[['region','dialect_sentence','standard_bengali']].to_csv(OUT/'audit'/'final_clean_dataset.csv',index=False,encoding='utf-8-sig')
print('FINAL CLEAN FILE:',OUT/'audit'/'final_clean_dataset.csv')
print(quality);print(df.region.value_counts())


FINAL CLEAN FILE: /content/drive/MyDrive/BANGLA DIALECT PAPER/DATASET/reviewer_ready_v8_reviewer_complete/audit/final_clean_dataset.csv
{'raw_rows': 23440, 'invalid_or_unmapped': 0, 'exact_triplet_duplicates': 406, 'ambiguous_input_strings': 455, 'outside_4_60_tokens': 3341, 'clean_rows': 18650}
region
Noakhali      3781
Barisal       3743
Khulna        3742
Sylhet        3698
Chittagong    3686
Name: count, dtype: int64


Markdown cell 11
Cell 11 of 60
Leakage-aware grouping BEFORE split
Union exact matching input or reference, plus high-similarity character-TFIDF pairs on inputs and references. The threshold is conservative and can merge genuine dialect variants. This code audits near pairs, and keeps them in the same group rather than deleting them. Sparse chunked cosine calculations are expensive but do not silently cap candidate counts.
Code cell 10

In [ ]:
class DSU:
    def __init__(self,n):self.p=list(range(n));self.size=[1]*n
    def find(self,x):
        while self.p[x]!=x:
            self.p[x]=self.p[self.p[x]];x=self.p[x]
        return x
    def union(self,a,b):
        a,b=self.find(a),self.find(b)
        if a==b:return
        if self.size[a]<self.size[b]:a,b=b,a
        self.p[b]=a;self.size[a]+=self.size[b]

def exact_union(values,dsu):
    seen={}
    for i,v in enumerate(values):
        if v in seen:dsu.union(i,seen[v])
        else:seen[v]=i

def near_pairs(values,threshold=.90,batch=192):
    # Unique strings first, then link their first indices; exact duplicates handled separately.
    first={}
    for i,v in enumerate(values):first.setdefault(v,i)
    strings=list(first); indices=[first[s] for s in strings]
    vec=TfidfVectorizer(analyzer='char',ngram_range=NEAR_NGRAM,sublinear_tf=True,dtype=np.float32)
    X=vec.fit_transform(strings)
    for st in range(0,len(strings),batch):
        S=(X[st:st+batch] @ X.T).tocoo()
        for r,c,v in zip(S.row,S.col,S.data):
            a=st+int(r); c=int(c)
            if a<c and v>=threshold:
                yield indices[a],indices[c],float(v)

dsu=DSU(len(df))
for col in ['dialect_sentence','standard_bengali']:exact_union(df[col].tolist(),dsu)
near_audit=[]
for col in ['dialect_sentence','standard_bengali']:
    print('Grouping near-duplicate',col)
    count=0
    for a,b,sim in near_pairs(df[col].tolist(),NEAR_THRESHOLD):
        count+=1
        if len(near_audit)<1000000:near_audit.append((col,a,b,sim))
        dsu.union(a,b)
    print('Pairs >= threshold:',count)
pd.DataFrame(near_audit,columns=['column','row_a','row_b','cosine']).to_csv(OUT/'audit'/'internal_near_duplicate_pairs.csv',index=False)
# Convert list to numpy array to meet pandas 3.x factorize input constraints
group_list = np.array([dsu.find(i) for i in range(len(df))])
df['group_id']=pd.factorize(group_list)[0]
print('Connected groups:',df.group_id.nunique(),'largest:',df.groupby('group_id').size().max())

Grouping near-duplicate dialect_sentence
Pairs >= threshold: 420
Grouping near-duplicate standard_bengali
Pairs >= threshold: 106
Connected groups: 3826 largest: 26


4. Leakage-aware grouping BEFORE split
   Union exact matching input or reference, plus high-similarity character-TFIDF pairs on inputs and references. The threshold is conservative and can merge genuine dialect variants. This code audits near pairs, and keeps them in the same group rather than deleting them. Sparse chunked cosine calculations are expensive but do not silently cap candidate counts.
   Code cell 10


Code cell 12

In [ ]:
class DSU:
    def __init__(self,n):self.p=list(range(n));self.size=[1]*n
    def find(self,x):
        while self.p[x]!=x:
            self.p[x]=self.p[self.p[x]];x=self.p[x]
        return x
    def union(self,a,b):
        a,b=self.find(a),self.find(b)
        if a==b:return
        if self.size[a]<self.size[b]:a,b=b,a
        self.p[b]=a;self.size[a]+=self.size[b]

def exact_union(values,dsu):
    seen={}
    for i,v in enumerate(values):
        if v in seen:dsu.union(i,seen[v])
        else:seen[v]=i

def near_pairs(values,threshold=.90,batch=192):
    # Unique strings first, then link their first indices; exact duplicates handled separately.
    first={}
    for i,v in enumerate(values):first.setdefault(v,i)
    strings=list(first); indices=[first[s] for s in strings]
    vec=TfidfVectorizer(analyzer='char',ngram_range=NEAR_NGRAM,sublinear_tf=True,dtype=np.float32)
    X=vec.fit_transform(strings)
    for st in range(0,len(strings),batch):
        S=(X[st:st+batch] @ X.T).tocoo()
        for r,c,v in zip(S.row,S.col,S.data):
            a=st+int(r); c=int(c)
            if a<c and v>=threshold:
                yield indices[a],indices[c],float(v)

dsu=DSU(len(df))
for col in ['dialect_sentence','standard_bengali']:exact_union(df[col].tolist(),dsu)
near_audit=[]
for col in ['dialect_sentence','standard_bengali']:
    print('Grouping near-duplicate',col)
    count=0
    for a,b,sim in near_pairs(df[col].tolist(),NEAR_THRESHOLD):
        count+=1
        if len(near_audit)<1000000:near_audit.append((col,a,b,sim))
        dsu.union(a,b)
    print('Pairs >= threshold:',count)
pd.DataFrame(near_audit,columns=['column','row_a','row_b','cosine']).to_csv(OUT/'audit'/'internal_near_duplicate_pairs.csv',index=False)
# Convert list to numpy array to meet pandas 3.x factorize input constraints
group_list = np.array([dsu.find(i) for i in range(len(df))])
df['group_id']=pd.factorize(group_list)[0]
print('Connected groups:',df.group_id.nunique(),'largest:',df.groupby('group_id').size().max())

Grouping near-duplicate dialect_sentence
Pairs >= threshold: 420
Grouping near-duplicate standard_bengali
Pairs >= threshold: 106
Connected groups: 3826 largest: 26


5. Group-disjoint 70/15/15 split with deterministic greedy balancing and hard integrity assertions
   Code cell 12


Code cell 14

In [ ]:
# Entire connected groups are indivisible; target ratios are approximate.
# Greedy group assignment with class-aware target and multiple deterministic restarts.
G=df.groupby('group_id').region.value_counts().unstack(fill_value=0).reindex(columns=LABELS,fill_value=0)
counts=G.to_numpy(dtype=int); groups=G.index.to_numpy()
target=np.array([TRAIN_FRAC,VAL_FRAC,TEST_FRAC])[:,None]*counts.sum(axis=0)[None,:]

def allocate(seed):
    rng=np.random.default_rng(seed)
    order=list(range(len(groups)));rng.shuffle(order)
    order.sort(key=lambda i:(-counts[i].sum(),-counts[i].max()))
    tally=np.zeros((3,len(LABELS)),dtype=int);assign={}
    for i in order:
        v=counts[i]
        scores=[]
        for k in range(3):
            nxt=tally.copy();nxt[k]+=v
            scores.append(float(np.square((nxt-target)/np.maximum(target,1)).sum()))
        k=int(np.argmin(scores));tally[k]+=v;assign[groups[i]]=k
    return assign,tally,float(np.square((tally-target)/np.maximum(target,1)).sum())
choices=[allocate(SEED+i) for i in range(20)]
assign,tally,_=min(choices,key=lambda x:x[2])
# If giant components make coverage impossible, fail rather than report an invalid stratified split.
if (tally==0).any():raise RuntimeError('Group structure prevents all classes in all splits. Review connected components; do not break groups.')
df['split']=df.group_id.map(assign).map({0:'train',1:'val',2:'test'})
splits={k:df[df.split==k].copy().reset_index(drop=True) for k in ['train','val','test']}
for col in ['group_id','dialect_sentence','standard_bengali']:
    for a,b in [('train','val'),('train','test'),('val','test')]:
        assert not (set(splits[a][col]) & set(splits[b][col])),f'Leakage: {col}, {a} vs {b}'
for k,z in splits.items():z.to_csv(OUT/'splits'/f'{k}.csv',index=False)
print(pd.crosstab(df.region,df.split));print('Fractions:',{k:round(len(v)/len(df),4) for k,v in splits.items()})
(OUT/'audit'/'split_summary.json').write_text(json.dumps({k:{'rows':len(v),'labels':v.region.value_counts().to_dict()} for k,v in splits.items()},indent=2),encoding='utf-8')


split       test  train  val
region                      
Barisal      564   2615  564
Chittagong   556   2575  555
Khulna       561   2620  561
Noakhali     570   2641  570
Sylhet       557   2585  556
Fractions: {'train': 0.699, 'val': 0.1505, 'test': 0.1506}


512

6. External 1/2/3 prepare from original source; strict exact and near-duplicate disjointness
   Code cell 14


Code cell 16

In [ ]:
def prep(z,name):
    z=z.copy()
    for col in ['region','dialect_sentence','standard_bengali']:z[col]=z[col].map(norm)
    z['region']=z.region.map(lambda x:ALIASES.get(x.lower(),x))
    z=z[z.region.isin(LABELS)&z.dialect_sentence.ne('')&z.standard_bengali.ne('')].copy()
    z=z.drop_duplicates(['region','dialect_sentence','standard_bengali']).reset_index(drop=True)
    z['source_dataset']=name
    return z[['region','dialect_sentence','standard_bengali','source_dataset']]

def wide_to_long(raw,reference,mapping,source):
    out=[]
    for col,region in mapping.items():
        if col not in raw:raise KeyError(f'{source}: {col} missing')
        out.append(pd.DataFrame({'region':region,'dialect_sentence':raw[col],'standard_bengali':raw[reference]}))
    return prep(pd.concat(out,ignore_index=True),source)

# Helper to dynamically find correct file under EXT_ROOT or its parent 'external'
def find_external_file(pattern):
    matches = list(EXT_ROOT.rglob(pattern))
    if not matches:
        matches = list(EXT_ROOT.parent.rglob(pattern))
    if not matches:
        raise FileNotFoundError(f"Could not dynamically find file matching '{pattern}' in your Drive folders.")
    return matches[0]

e1raw=pd.read_excel(find_external_file('BanglaRegionalTextCorpus-tnQRMn.xlsx'))
e1=prep(e1raw.rename(columns={'Region_Labels':'region','Regional_Texts':'dialect_sentence','Standard_Bangla_Texts':'standard_bengali'}),'External1')

e2raw=pd.read_csv(find_external_file('BD-Dialect_Clauses.csv'))
e2=wide_to_long(e2raw,'Standard Bangla Languge',{'Nowakhali Language':'Noakhali','Sylheti Language':'Sylhet','Chittagong Language':'Chittagong'},'External2')

e3raw=pd.read_excel(find_external_file('Dwadash.xlsx'))
e3=wide_to_long(e3raw,'bangla_speech',{'sylhet_bangla_speech':'Sylhet','barishal_bangla_speech':'Barisal','chittagong_bangla_speech':'Chittagong','noakhali_bangla_speech':'Noakhali'},'External3')

external_raw={'External1':e1,'External2':e2,'External3':e3}
for name,z in external_raw.items():
    z.to_csv(OUT/'prepared_external'/f'{name}_before_audit.csv',index=False)
    print(name,len(z),z.region.value_counts().to_dict())

External1 2052 {'Barisal': 1135, 'Khulna': 917}
External2 2845 {'Chittagong': 949, 'Noakhali': 948, 'Sylhet': 948}
External3 26982 {'Chittagong': 10495, 'Sylhet': 7302, 'Noakhali': 4998, 'Barisal': 4187}


Strict external overlap filtering
For each external dataset, remove any row whose input or reference exactly matches any internal train/val/test input or reference, or is near-duplicate (character TF-IDF cosine ≥ threshold) of internal input/reference. Also remove cross-external exact repeats from later datasets to prevent double counting. Refit audit vectorizer on combined unlabeled strings only (audit, never model training). No reference text is fed to the classifier.

Code cell 16


Code cell 18

In [ ]:
internal_all=pd.concat(splits.values(),ignore_index=True)
internal_inputs=set(internal_all.dialect_sentence);internal_refs=set(internal_all.standard_bengali)

def near_hit_mask(external_values,internal_values,threshold=NEAR_THRESHOLD,batch=96):
    # All external strings scored against all internal strings, no top-k truncation.
    iv=list(set(internal_values));ev=list(dict.fromkeys(external_values))
    if not ev:return np.zeros(len(external_values),dtype=bool),np.zeros(len(external_values))
    vec=TfidfVectorizer(analyzer='char',ngram_range=NEAR_NGRAM,sublinear_tf=True,dtype=np.float32)
    X=vec.fit_transform(iv+ev); A=X[:len(iv)];B=X[len(iv):]
    mx=np.zeros(len(ev),dtype=np.float32)
    for st in range(0,len(ev),batch):
        sim=(B[st:st+batch]@A.T).tocsr()
        mx[st:st+batch]=np.asarray(sim.max(axis=1).toarray()).ravel()
    lookup=dict(zip(ev,mx))
    vals=np.array([lookup[v] for v in external_values]);return vals>=threshold,vals

external_clean={};audit=[];seen_inputs=set();seen_refs=set()
for name,z in external_raw.items():
    print('Auditing',name,'against ALL internal splits...')
    a=z.dialect_sentence.tolist();r=z.standard_bengali.tolist()
    exact_i=np.array([v in internal_inputs for v in a]);exact_r=np.array([v in internal_refs for v in r])
    near_i,sim_i=near_hit_mask(a,internal_inputs)
    near_r,sim_r=near_hit_mask(r,internal_refs)
    cross_ext=np.array([(x in seen_inputs) or (y in seen_refs) for x,y in zip(a,r)])
    # Both exact and near are included; external datasets are processed in fixed order.
    drop=exact_i|exact_r|near_i|near_r|cross_ext
    report=z.copy();report['exact_input_internal']=exact_i;report['exact_ref_internal']=exact_r
    report['near_input_internal']=near_i;report['near_ref_internal']=near_r
    report['max_input_cosine']=sim_i;report['max_ref_cosine']=sim_r
    report['cross_external_exact']=cross_ext;report['excluded']=drop
    report.to_csv(OUT/'audit'/f'{name}_row_overlap_audit.csv',index=False)
    keep=z.loc[~drop].reset_index(drop=True)
    external_clean[name]=keep
    keep.to_csv(OUT/'prepared_external'/f'{name}_strict_disjoint.csv',index=False)
    seen_inputs.update(keep.dialect_sentence);seen_refs.update(keep.standard_bengali)
    audit.append({'dataset':name,'prepared':len(z),'exact_input_internal':int(exact_i.sum()),'exact_ref_internal':int(exact_r.sum()),
                  'near_input_internal':int(near_i.sum()),'near_ref_internal':int(near_r.sum()),
                  'cross_external_exact':int(cross_ext.sum()),'excluded_union':int(drop.sum()),'remaining':len(keep)})
audit_df=pd.DataFrame(audit);audit_df.to_csv(OUT/'audit'/'external_leakage_summary.csv',index=False)
display(audit_df)


Auditing External1 against ALL internal splits...
Auditing External2 against ALL internal splits...
Auditing External3 against ALL internal splits...


,dataset,prepared,exact_input_internal,exact_ref_internal,near_input_internal,near_ref_internal,cross_external_exact,excluded_union,remaining
0,External1,2052,97,10,110,29,0,130,1922
1,External2,2845,0,0,0,0,0,0,2845
2,External3,26982,10473,12836,10577,12900,79,12986,13996


Code cell 19

In [ ]:
# REVIEWER PRE-FLIGHT: dataset/split fingerprint and safe reuse policy
import hashlib, json
from pathlib import Path
manifest_path = OUT / 'audit' / 'run_manifest.json'
def _split_digest(parts):
    h = hashlib.sha256()
    for key in ('train','val','test'):
        frame = parts[key]
        for col in ('region','dialect_sentence','standard_bengali'):
            h.update(key.encode());h.update(col.encode())
            for value in frame[col].fillna('').astype(str):
                h.update(value.encode('utf-8'));h.update(b'\x00')
    return h.hexdigest()
current_digest = _split_digest(splits)
if manifest_path.exists():
    old = json.loads(manifest_path.read_text(encoding='utf-8'))
    if old.get('split_sha256') != current_digest:
        raise RuntimeError('Dataset/split changed for this output directory. Use a new OUT folder or archive old results; do not mix runs.')
else:
    manifest_path.parent.mkdir(parents=True,exist_ok=True)
    manifest_path.write_text(json.dumps({'split_sha256':current_digest,
        'split_counts':{k:len(v) for k,v in splits.items()},
        'seed':SEED, 'train_fraction_target':TRAIN_FRAC,
        'validation_fraction_target':VAL_FRAC,'test_fraction_target':TEST_FRAC},
        indent=2),encoding='utf-8')
print('Run fingerprint:',current_digest[:16])
print('All outputs use stable names; reruns overwrite existing CSV/PNG rather than creating copies.')


Run fingerprint: 2b4662b55002e7b7
All outputs use stable names; reruns overwrite existing CSV/PNG rather than creating copies.


7. Shared evaluation and 300 DPI figure helpers
   Code cell 18


Code cell 21

In [ ]:
complexity_records=[]
from sklearn.metrics import ConfusionMatrixDisplay

def savefig(name):
    plt.tight_layout()
    plt.savefig(OUT/'figures'/f'{name}.png',dpi=300,bbox_inches='tight')
    plt.close()

def eval_classifier(name,model_name,part,y_true,pred,proba=None):
    y=np.array([L2I[v] for v in y_true]);yp=np.array(pred,dtype=int)
    report=classification_report(y,yp,labels=list(range(5)),target_names=LABELS,output_dict=True,zero_division=0)
    stem=f'{model_name}_{part}'
    pd.DataFrame(report).T.to_csv(OUT/'metrics'/f'{stem}_classification_report.csv')
    cm=confusion_matrix(y,yp,labels=list(range(5)))
    pd.DataFrame(cm,index=LABELS,columns=LABELS).to_csv(OUT/'metrics'/f'{stem}_confusion_matrix.csv')
    fig,ax=plt.subplots(figsize=(7,6));ConfusionMatrixDisplay(cm,display_labels=LABELS).plot(ax=ax,cmap='Blues',xticks_rotation=40,colorbar=False)
    ax.set_title(f'{model_name} — {part}');savefig(f'{stem}_confusion_matrix')
    row={'task':'classification','model':model_name,'dataset':part,'n':len(y),
         'accuracy':accuracy_score(y,yp),'macro_f1':f1_score(y,yp,labels=list(range(5)),average='macro',zero_division=0),
         'covered_class_macro_f1':f1_score(y,yp,labels=sorted(set(y)),average='macro',zero_division=0),
         'gold_classes_present':','.join(sorted(set(y_true))),
         'weighted_f1':f1_score(y,yp,average='weighted',zero_division=0),
         'macro_precision':precision_recall_fscore_support(y,yp,labels=list(range(5)),average='macro',zero_division=0)[0],
         'macro_recall':precision_recall_fscore_support(y,yp,labels=list(range(5)),average='macro',zero_division=0)[1]}
    if proba is not None:
        p=np.asarray(proba)
        if p.shape==(len(y),5):
            # AUC only over evaluable one-vs-rest classes; avoid undefined ROC on partial-class external data.
            auc_by_class={}
            for k,label in enumerate(LABELS):
                binary=(y==k).astype(int)
                if np.unique(binary).size==2:
                    auc_by_class[label]=float(roc_auc_score(binary,p[:,k]))
            row['macro_ovr_auc']=float(np.mean(list(auc_by_class.values()))) if auc_by_class else np.nan
            row['auc_evaluable_classes']=len(auc_by_class)
            row['auc_definition']='mean valid one-vs-rest class AUCs; not full 5-class macro AUC'
            fig,ax=plt.subplots(figsize=(7,6))
            for k,label in enumerate(LABELS):
                binary=(y==k).astype(int)
                if len(np.unique(binary))<2:continue
                fpr,tpr,_=roc_curve(binary,p[:,k]);ax.plot(fpr,tpr,label=f'{label} AUC={auc(fpr,tpr):.3f}')
            ax.plot([0,1],[0,1],'k--',lw=1);ax.set(xlabel='False positive rate',ylabel='True positive rate',title=f'{model_name} ROC — {part}')

            if auc_by_class: ax.legend(fontsize=8)
            savefig(f'{stem}_roc')
    pd.DataFrame({'true':y_true,'predicted':[I2L[k] for k in yp]}).to_csv(OUT/'predictions'/f'{stem}.csv',index=False)
    return row

# Idempotent metric store: rerunning cells replaces the same model/dataset result.
metric_rows=[]
def add_metric(row):
    """Idempotent CSV upsert, keyed by task/model/dataset."""
    path = OUT / 'metrics' / 'all_results.csv'
    path.parent.mkdir(parents=True, exist_ok=True)
    fresh = pd.DataFrame([row])
    if path.exists():
        previous = pd.read_csv(path)
        combined = pd.concat([previous, fresh], ignore_index=True)
    else:
        combined = fresh
    combined = combined.drop_duplicates(subset=['task', 'model', 'dataset'], keep='last')
    combined = combined.sort_values(['task', 'model', 'dataset']).reset_index(drop=True)
    combined.to_csv(path, index=False)
    metric_rows[:] = combined.to_dict('records')


8. Classical models: training, validation learning curves, held-out test and all external datasets
   Code cell 20


Code cell 23

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.base import clone

classical={
 'LogisticRegression':LogisticRegression(max_iter=2000,class_weight='balanced',random_state=SEED),
 'LinearSVM':LinearSVC(class_weight='balanced',random_state=SEED,max_iter=4000),
 'MultinomialNB':MultinomialNB(),
 'RandomForest':RandomForestClassifier(n_estimators=300,class_weight='balanced',n_jobs=-1,random_state=SEED),
}

def classical_pipe(est):return Pipeline([('tfidf',TfidfVectorizer(analyzer='char',ngram_range=(2,5),min_df=2,sublinear_tf=True,max_features=100000)),('clf',clone(est))])

if RUN_CLASSICAL:
    tr=splits['train'];va=splits['val'];te=splits['test']
    for name,est in classical.items():
        print('Training',name)
        model=classical_pipe(est)
        _t0=time.perf_counter();model.fit(tr.dialect_sentence,tr.region.map(L2I));_train_sec=time.perf_counter()-_t0
        _t1=time.perf_counter();_ = model.predict(te.dialect_sentence);_infer_sec=time.perf_counter()-_t1
        complexity_records.append(dict(task="classification",family="Classical ML",model=name,device="CPU",train_seconds=_train_sec,inference_ms_per_sample=1000*_infer_sec/max(1,len(te)),parameters=np.nan,gpu_peak_mb=np.nan))
        joblib.dump(model,OUT/'models'/f'{name}.joblib')
        # Learning curves use train subsets and untouched validation, not held-out test.
        curves=[]
        for frac in [.10,.25,.50,.75,1.0]:
            sub=tr.groupby('region',group_keys=False).sample(frac=frac,random_state=SEED) if frac<1 else tr
            m=classical_pipe(est);m.fit(sub.dialect_sentence,sub.region.map(L2I))
            for part,d in [('train_subset',sub),('validation',va)]:
                yp=m.predict(d.dialect_sentence)
                curves.append({'fraction':frac,'n_train':len(sub),'partition':part,'macro_f1':f1_score(d.region.map(L2I),yp,average='macro',zero_division=0)})
        curve_df=pd.DataFrame(curves);curve_df.to_csv(OUT/'metrics'/f'{name}_learning_curve.csv',index=False)
        fig,ax=plt.subplots(figsize=(7,5))
        for part,g in curve_df.groupby('partition'):ax.plot(g.n_train,g.macro_f1,'o-',label=part)
        ax.set(xlabel='Number of training samples',ylabel='Macro-F1',title=f'{name} learning curve');ax.legend();savefig(f'{name}_learning_curve')
        for part,d in [('validation',va),('internal_test',te)]+list(external_clean.items()):
            if len(d)==0:continue
            pred=model.predict(d.dialect_sentence)
            # SVM decision scores are converted via softmax for plotting ROC ranks only;
            # they are NOT calibrated probabilities. AUC uses ranking and remains meaningful.
            if hasattr(model,'predict_proba'):
                p=model.predict_proba(d.dialect_sentence)
            elif hasattr(model,'decision_function'):
                p=softmax(model.decision_function(d.dialect_sentence),axis=1)
            else:p=None
            add_metric(eval_classifier(name,name,part,d.region,pred,p))
        del model;gc.collect()


Training LogisticRegression
Training LinearSVM
Training MultinomialNB
Training RandomForest


9. Transformer classification — BanglaBERT, mBERT, MuRIL, XLM-R (validation-selected checkpoint)
   Code cell 22


Code cell 25

In [ ]:
from datasets import Dataset
from transformers import (AutoTokenizer,AutoModelForSequenceClassification,Trainer,TrainingArguments,DataCollatorWithPadding)

CLS_MODELS={
 'BanglaBERT':'sagorsarker/bangla-bert-base',
 'mBERT':'bert-base-multilingual-cased',
 'MuRIL':'google/muril-base-cased',
 'XLM-R':'xlm-roberta-base',
}

def cls_ds(z,tokenizer):
    ds=Dataset.from_dict({'text':z.dialect_sentence.tolist(),'labels':z.region.map(L2I).astype(int).tolist()})
    return ds.map(lambda b:tokenizer(b['text'],truncation=True,max_length=128),batched=True,remove_columns=['text'])

def cls_metrics(out):
    y=out.label_ids;yp=out.predictions.argmax(axis=1)
    return {'accuracy':accuracy_score(y,yp),'macro_f1':f1_score(y,yp,average='macro',zero_division=0)}

if RUN_TRANSFORMERS:
    assert torch.cuda.is_available(),'GPU required for transformer training; enable GPU or set RUN_TRANSFORMERS=False'
    for name,backbone in CLS_MODELS.items():
        print('TRAIN:',name,backbone)
        ck=OUT/'models'/name
        tok=AutoTokenizer.from_pretrained(backbone)
        model=AutoModelForSequenceClassification.from_pretrained(backbone,num_labels=5,id2label=I2L,label2id=L2I,ignore_mismatched_sizes=True)
        tr_ds=cls_ds(splits['train'],tok);va_ds=cls_ds(splits['val'],tok)
        args=TrainingArguments(output_dir=str(OUT/'logs'/name),num_train_epochs=CLASSIFIER_EPOCHS,
             per_device_train_batch_size=CLASSIFIER_BATCH,per_device_eval_batch_size=CLASSIFIER_BATCH,
             learning_rate=2e-5,weight_decay=.01,eval_strategy='epoch',save_strategy='epoch',
             load_best_model_at_end=True,metric_for_best_model='macro_f1',greater_is_better=True,
             save_total_limit=2,logging_strategy='epoch',report_to='none',seed=SEED,fp16=False)
        trainer=Trainer(model=model,args=args,train_dataset=tr_ds,eval_dataset=va_ds,
             processing_class=tok,data_collator=DataCollatorWithPadding(tok),compute_metrics=cls_metrics)
        if torch.cuda.is_available():torch.cuda.reset_peak_memory_stats()
        _t0=time.perf_counter();trainer.train();_train_sec=time.perf_counter()-_t0
        _peak=torch.cuda.max_memory_allocated()/1024**2 if torch.cuda.is_available() else np.nan
        _params=sum(p.numel() for p in trainer.model.parameters())
        complexity_records.append(dict(task="classification",family="Transformer",model=name,device="GPU" if torch.cuda.is_available() else "CPU",train_seconds=_train_sec,inference_ms_per_sample=np.nan,parameters=_params,gpu_peak_mb=_peak))
        trainer.save_model(str(ck));tok.save_pretrained(str(ck))
        history=pd.DataFrame(trainer.state.log_history);history.to_csv(OUT/'logs'/f'{name}_history.csv',index=False)
        fig,ax=plt.subplots(figsize=(7,5))
        for metric in ['loss','eval_loss']:
            h=history.dropna(subset=[metric]) if metric in history else pd.DataFrame()
            if len(h):ax.plot(h.epoch,h[metric],marker='o',label=metric)
        ax.set(xlabel='Epoch',ylabel='Loss',title=f'{name} training / validation loss');ax.legend();savefig(f'{name}_learning_curve')
        for part,d in [('validation',splits['val']),('internal_test',splits['test'])]+list(external_clean.items()):
            if len(d)==0:continue
            pred_out=trainer.predict(cls_ds(d,tok));prob=softmax(pred_out.predictions,axis=1)
            add_metric(eval_classifier(name,name,part,d.region,prob.argmax(axis=1),prob))
        del trainer,model,tok,tr_ds,va_ds;gc.collect();torch.cuda.empty_cache()


TRAIN: BanglaBERT sagorsarker/bangla-bert-base


config.json:   0%|          | 0.00/491 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/660M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at sagorsarker/bangla-bert-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Map:   0%|          | 0/13036 [00:00<?, ? examples/s]

... (49 outputs left out)

Map:   0%|          | 0/2808 [00:00<?, ? examples/s]

Map:   0%|          | 0/1922 [00:00<?, ? examples/s]

Map:   0%|          | 0/2845 [00:00<?, ? examples/s]

Map:   0%|          | 0/13996 [00:00<?, ? examples/s]

9B. Classification results — internal test, external reports, 300 DPI comparison figures

Code cell 27


Code cell 27

In [ ]:
# Run AFTER Sections 8 and 9. No training or model reload required.
from pathlib import Path
results_file=OUT/'metrics'/'all_results.csv'
assert results_file.exists(), 'Train/evaluate classification models first (Sections 8 and/or 9).'
cls_results=pd.read_csv(results_file)
cls_results=cls_results[cls_results['task']=='classification'].copy()
assert not cls_results.empty, 'No classification results saved yet.'
cols=['model','dataset','n','accuracy','macro_precision','macro_recall','macro_f1',
      'covered_class_macro_f1','weighted_f1','macro_ovr_auc','gold_classes_present']
cols=[c for c in cols if c in cls_results]
cls_results[cols].to_csv(OUT/'metrics'/'classification_all_models_detailed.csv',index=False)
display(cls_results[cols].sort_values(['dataset','macro_f1'],ascending=[True,False]))
for part,z in cls_results.groupby('dataset'):
    z=z.sort_values('macro_f1',ascending=False)
    fig,ax=plt.subplots(figsize=(max(9,len(z)*1.2),5))
    x=np.arange(len(z));width=.38
    ax.bar(x-width/2,z.accuracy,width,label='Accuracy')
    ax.bar(x+width/2,z.macro_f1,width,label='5-class Macro-F1')
    ax.set_xticks(x,labels=z.model,rotation=30,ha='right')
    ax.set(ylim=(0,1.05),ylabel='Score',title=f'Classification model comparison: {part}')
    ax.legend();ax.grid(axis='y',alpha=.2)
    savefig('classification_comparison_'+str(part))
print('Detailed reports and 300 DPI classification comparison figures saved in',OUT)


,model,dataset,n,accuracy,macro_precision,macro_recall,macro_f1,covered_class_macro_f1,weighted_f1,macro_ovr_auc,gold_classes_present
20,MultinomialNB,External1,1922,0.664932,0.319823,0.262560,0.281597,0.703994,0.708135,0.834209,"Barisal,Khulna"
5,LinearSVM,External1,1922,0.661811,0.327795,0.260266,0.276987,0.692468,0.697759,0.828811,"Barisal,Khulna"
10,LogisticRegression,External1,1922,0.641519,0.317699,0.252891,0.273033,0.682582,0.687223,0.826224,"Barisal,Khulna"
30,XLM-R,External1,1922,0.629553,0.325933,0.247681,0.268033,0.670081,0.674861,0.837261,"Barisal,Khulna"
15,MuRIL,External1,1922,0.631113,0.321999,0.248296,0.266616,0.666541,0.671145,0.781660,"Barisal,Khulna"
35,mBERT,External1,1922,0.635796,0.320614,0.249410,0.263653,0.659133,0.665122,0.812134,"Barisal,Khulna"
0,BanglaBERT,External1,1922,0.607700,0.324286,0.238805,0.261518,0.653796,0.659325,0.815229,"Barisal,Khulna"
25,RandomForest,External1,1922,0.560874,0.292037,0.221062,0.245274,0.613184,0.618023,0.781268,"Barisal,Khulna"
16,MuRIL,External2,2845,0.456942,0.377299,0.274111,0.310267,0.517111,0.517178,0.739158,"Chittagong,Noakhali,Sylhet"
31,XLM-R,External2,2845,0.436204,0.356710,0.261687,0.296859,0.494765,0.494813,0.721232,"Chittagong,Noakhali,Sylhet"


Detailed reports and 300 DPI classification comparison figures saved in /content/drive/MyDrive/BANGLA DIALECT PAPER/DATASET/reviewer_ready_v8_reviewer_complete


10. Standardization evaluation helpers
    Sentence-level BLEU uses sacrebleu.corpus_bleu for a corpus score, chrF uses corpus_chrf. No ROC or confusion matrix for text generation. Report separate word-level evaluation only if enabled.
    Code cell 24


Code cell 29

In [ ]:
import sacrebleu
from rouge_score import rouge_scorer
from nltk.translate.bleu_score import sentence_bleu,SmoothingFunction

def eval_generator(model_name,part,df,predictions):
    refs=[norm(x) for x in df.standard_bengali.astype(str).tolist()]
    preds=[norm(x) for x in predictions]
    assert len(preds)==len(refs)
    rouge=rouge_scorer.RougeScorer(['rougeL'],use_stemmer=False)
    rouge_l=float(np.mean([rouge.score(ref,pred)['rougeL'].fmeasure for ref,pred in zip(refs,preds)]))
    row={'task':'standardization','model':model_name,'dataset':part,'n':len(df),
         'bleu':sacrebleu.corpus_bleu(preds,[refs]).score,
         'chrf':sacrebleu.corpus_chrf(preds,[refs]).score,
         'exact_match':np.mean([a==b for a,b in zip(preds,refs)]), 'rouge_l':rouge_l,
         'empty_prediction_rate':float(np.mean([not p for p in preds])),
         'mean_prediction_tokens':float(np.mean([len(p.split()) for p in preds])),
         'mean_reference_tokens':float(np.mean([len(r.split()) for r in refs]))}
    pd.DataFrame({'region':df.region,'input':df.dialect_sentence,'reference':refs,'prediction':preds}).to_csv(
        OUT/'predictions'/f'{model_name}_{part}_generation.csv',index=False)
    add_metric(row)
    return row


11. LSTM encoder–decoder Seq2Seq (train-only vocabulary, teacher forcing, greedy decoding)
    Code cell 26


Code cell 31

In [ ]:
from torch import nn
from torch.utils.data import Dataset as TorchDataset,DataLoader
from collections import Counter

PAD,UNK,BOS,EOS=0,1,2,3

def build_vocab(texts,max_words=30000):
    cnt=Counter(token for t in texts for token in t.split())
    vocab={'<pad>':PAD,'<unk>':UNK,'<bos>':BOS,'<eos>':EOS}
    for word,_ in cnt.most_common(max_words-4):
        if word not in vocab:vocab[word]=len(vocab)
    return vocab

class PairDataset(TorchDataset):
    def __init__(self,df,src_vocab,tgt_vocab):self.df=df.reset_index(drop=True);self.s=src_vocab;self.t=tgt_vocab
    def __len__(self):return len(self.df)
    def __getitem__(self,i):
        r=self.df.iloc[i]
        s=[BOS]+[self.s.get(w,UNK) for w in r.dialect_sentence.split()][:GEN_MAX_INPUT-2]+[EOS]
        t=[BOS]+[self.t.get(w,UNK) for w in r.standard_bengali.split()][:GEN_MAX_TARGET-2]+[EOS]
        return torch.tensor(s),torch.tensor(t)

def pair_collate(batch):
    a,b=zip(*batch)
    return nn.utils.rnn.pad_sequence(a,batch_first=True,padding_value=PAD),nn.utils.rnn.pad_sequence(b,batch_first=True,padding_value=PAD)

class LSTMSeq2Seq(nn.Module):
    def __init__(self,n_src,n_tgt,emb=256,hid=384):
        super().__init__();self.enc_emb=nn.Embedding(n_src,emb,padding_idx=PAD);self.dec_emb=nn.Embedding(n_tgt,emb,padding_idx=PAD)
        self.enc=nn.LSTM(emb,hid,batch_first=True);self.dec=nn.LSTM(emb,hid,batch_first=True);self.fc=nn.Linear(hid,n_tgt)
    def encode(self,src):
        lengths=src.ne(PAD).sum(dim=1).cpu()
        packed=nn.utils.rnn.pack_padded_sequence(self.enc_emb(src),lengths,batch_first=True,enforce_sorted=False)
        _,state=self.enc(packed)
        return state
    def forward(self,src,tgt):
        state=self.encode(src)
        h,_=self.dec(self.dec_emb(tgt[:,:-1]),state)
        return self.fc(h)
    @torch.no_grad()
    def generate(self,src,max_len=GEN_MAX_TARGET):
        state=self.encode(src);tok=torch.full((src.size(0),1),BOS,device=src.device,dtype=torch.long)
        out=[];done=torch.zeros(src.size(0),device=src.device,dtype=torch.bool)
        for _ in range(max_len):
            h,state=self.dec(self.dec_emb(tok),state);tok=self.fc(h[:,-1]).argmax(-1,keepdim=True)
            tok=torch.where(done.unsqueeze(1),torch.full_like(tok,EOS),tok)
            out.append(tok);done|=(tok.squeeze(1)==EOS)
            if done.all():break
        return torch.cat(out,dim=1)

if RUN_LSTM:
    dev='cuda' if torch.cuda.is_available() else 'cpu'
    src_vocab=build_vocab(splits['train'].dialect_sentence)
    tgt_vocab=build_vocab(splits['train'].standard_bengali)
    inv_tgt={v:k for k,v in tgt_vocab.items()}
    (OUT/'models'/'LSTM_vocab.json').write_text(json.dumps({'source':src_vocab,'target':tgt_vocab},ensure_ascii=False),encoding='utf-8')
    lstm=LSTMSeq2Seq(len(src_vocab),len(tgt_vocab)).to(dev)
    opt=torch.optim.AdamW(lstm.parameters(),lr=.001)
    criterion=nn.CrossEntropyLoss(ignore_index=PAD)
    train_loader=DataLoader(PairDataset(splits['train'],src_vocab,tgt_vocab),batch_size=64,shuffle=True,collate_fn=pair_collate)
    val_loader=DataLoader(PairDataset(splits['val'],src_vocab,tgt_vocab),batch_size=64,shuffle=False,collate_fn=pair_collate)
    best=float('inf');history=[];patience=4;bad_epochs=0;start_lstm=time.perf_counter()
    for ep in range(1,LSTM_EPOCHS+1):
        losses=[];lstm.train()
        for src,tgt in train_loader:
            src,tgt=src.to(dev),tgt.to(dev);opt.zero_grad()
            logits=lstm(src,tgt);loss=criterion(logits.reshape(-1,logits.shape[-1]),tgt[:,1:].reshape(-1))
            loss.backward();nn.utils.clip_grad_norm_(lstm.parameters(),1.0);opt.step();losses.append(loss.item())
        lstm.eval();v=[]
        with torch.no_grad():
            for src,tgt in val_loader:
                src,tgt=src.to(dev),tgt.to(dev);logits=lstm(src,tgt)
                v.append(criterion(logits.reshape(-1,logits.shape[-1]),tgt[:,1:].reshape(-1)).item())
        vl=float(np.mean(v));history.append({'epoch':ep,'train_loss':np.mean(losses),'val_loss':vl})
        print('LSTM epoch',ep,'train',np.mean(losses),'val',vl)
        if vl<best:
            best=vl;bad_epochs=0;torch.save(lstm.state_dict(),OUT/'models'/'LSTM_best.pt')
        else:
            bad_epochs+=1
            if bad_epochs>=patience:
                print('LSTM early stopping');break
    lstm_train_seconds=time.perf_counter()-start_lstm
    hist=pd.DataFrame(history);hist.to_csv(OUT/'logs'/'LSTM_history.csv',index=False)
    fig,ax=plt.subplots(figsize=(7,5));ax.plot(hist.epoch,hist.train_loss,label='train');ax.plot(hist.epoch,hist.val_loss,label='validation');ax.legend();ax.set(xlabel='Epoch',ylabel='Loss',title='LSTM learning curve');savefig('LSTM_learning_curve')
    lstm.load_state_dict(torch.load(OUT/'models'/'LSTM_best.pt',map_location=dev,weights_only=True));lstm.eval()
    def lstm_predict(z):
        loader=DataLoader(PairDataset(z,src_vocab,tgt_vocab),batch_size=64,collate_fn=pair_collate)
        result=[]
        for src,_ in loader:
            ids=lstm.generate(src.to(dev)).cpu().tolist()
            for seq in ids:
                words=[]
                for i in seq:
                    if i==EOS:break
                    if i not in [PAD,BOS]:words.append(inv_tgt.get(i,'<unk>'))
                result.append(' '.join(words))
        return result
    for part,z in [('validation',splits['val']),('internal_test',splits['test'])]+list(external_clean.items()):
        if len(z):eval_generator('LSTM',part,z,lstm_predict(z))
    (OUT/'metrics'/'LSTM_compute.json').write_text(json.dumps({'model':'LSTM','parameters':sum(p.numel() for p in lstm.parameters()),'train_seconds':lstm_train_seconds,'device':dev}),encoding='utf-8')
    del lstm;gc.collect()
    if torch.cuda.is_available():torch.cuda.empty_cache()


LSTM epoch 1 train 5.7464569736929505 val 5.384556515650316
LSTM epoch 2 train 3.701174953404595 val 4.987838192419573
LSTM epoch 3 train 1.9633456491956525 val 4.880173753608357
LSTM epoch 4 train 1.0247410357582802 val 4.871998180042613
LSTM epoch 5 train 0.6006522986526582 val 4.886557709087025
LSTM epoch 6 train 0.3754357574324982 val 4.912424358454618
LSTM epoch 7 train 0.23137867384973695 val 4.938137612559578
LSTM epoch 8 train 0.13983453967261547 val 4.975897165861997
LSTM early stopping


12. Corrected transformer standardizers — BanglaT5, mT5-base, mBART-50
    Run one model at a time using GENERATOR_TO_RUN. This section uses fresh pretrained weights, a train-only fit, validation-only checkpoint selection, and the existing external_clean overlap-filtered sets. For the BanglaT5 diagnostics on this installation, use FP32 and gradient accumulation = 1; log training and validation losses separately. Figures are saved as 300-DPI PNG and PDF. bn_IN is the mBART-50 tokenizer language code for Bengali, not a dialect-origin assertion. Exact and threshold-based filtering cannot prove speaker/source or semantic disjointness.
    Code cell 28


Code cell 33

In [ ]:
# SECTION 12 — replace the previous generator training cell with this one.
# Requires previous cells: OUT, splits, external_clean, eval_generator, SEED,
# GEN_MAX_INPUT, GEN_MAX_TARGET, GENERATOR_EPOCHS, RUN_GENERATORS.
import gc, inspect, json, time
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import transformers
from datasets import Dataset
from transformers import (AutoTokenizer, AutoConfig, AutoModelForSeq2SeqLM,
    Seq2SeqTrainer, Seq2SeqTrainingArguments, DataCollatorForSeq2Seq)

GEN_MODELS = {
    'BanglaT5': 'csebuetnlp/banglat5',
    'mT5': 'google/mt5-base',
    'mBART50': 'facebook/mbart-large-50-many-to-many-mmt',
}
# Run one model per Colab execution. Change to 'mT5' or 'mBART50' later.
GENERATOR_TO_RUN = 'BanglaT5'
GENERATOR_BATCH_SIZE = 2 if GENERATOR_TO_RUN == 'BanglaT5' else 1
GENERATOR_GRAD_ACCUM = 1  # Avoid observed accumulation-dependent loss log scaling
GENERATOR_LR = 1e-5
GENERATOR_EVAL_BATCH = 2 if GENERATOR_TO_RUN == 'BanglaT5' else 1
GENERATOR_BEAMS = 4

for folder in ['models', 'logs', 'figures', 'predictions', 'metrics']:
    (OUT / folder).mkdir(parents=True, exist_ok=True)

assert GENERATOR_TO_RUN in GEN_MODELS
assert all(k in splits for k in ('train', 'val', 'test'))
assert len(splits['train']) and len(splits['val']) and len(splits['test'])
for label, frame in [('train', splits['train']), ('val', splits['val']), ('test', splits['test'])] + list(external_clean.items()):
    assert {'dialect_sentence', 'standard_bengali'}.issubset(frame.columns), label

# Fail closed on exact overlap; preceding notebook audit handles near-duplicate grouping.
def _norm_text(x):
    import unicodedata
    return unicodedata.normalize('NFC', str(x)).strip()
internal = pd.concat([splits[k] for k in ('train', 'val', 'test')], ignore_index=True)
for col in ('dialect_sentence', 'standard_bengali'):
    for a, b in [('train','val'), ('train','test'), ('val','test')]:
        assert set(splits[a][col].map(_norm_text)).isdisjoint(
            set(splits[b][col].map(_norm_text))), f'Internal overlap: {col} {a}/{b}'
    refset = set(internal[col].map(_norm_text))
    for name, frame in external_clean.items():
        assert set(frame[col].map(_norm_text)).isdisjoint(refset), f'External overlap: {name}/{col}'


def _make_seq_dataset(frame, tokenizer):
    frame = frame.dropna(subset=['dialect_sentence', 'standard_bengali'])
    source = frame['dialect_sentence'].astype(str).tolist()
    target = frame['standard_bengali'].astype(str).tolist()
    assert all(s.strip() and t.strip() for s,t in zip(source,target)), 'Blank source/target detected'
    data = Dataset.from_dict({'source': source, 'target': target})
    def encode(batch):
        x = tokenizer(batch['source'], truncation=True, max_length=GEN_MAX_INPUT)
        y = tokenizer(text_target=batch['target'], truncation=True, max_length=GEN_MAX_TARGET)
        x['labels'] = y['input_ids']
        return x
    return data.map(encode, batched=True, remove_columns=['source','target'])


def _make_args(name):
    accepted = inspect.signature(Seq2SeqTrainingArguments.__init__).parameters
    opts = dict(
        output_dir=str(OUT/'logs'/f'{name}_checkpoints'),
        num_train_epochs=GENERATOR_EPOCHS,
        per_device_train_batch_size=GENERATOR_BATCH_SIZE,
        per_device_eval_batch_size=GENERATOR_EVAL_BATCH,
        gradient_accumulation_steps=GENERATOR_GRAD_ACCUM,
        learning_rate=GENERATOR_LR,
        weight_decay=0.01,
        max_grad_norm=1.0,
        logging_strategy='epoch',
        save_strategy='epoch',
        load_best_model_at_end=True,
        metric_for_best_model='eval_loss',
        greater_is_better=False,
        save_total_limit=2,
        report_to='none',
        seed=SEED,
        fp16=False,
        bf16=False,
    )
    if 'eval_strategy' in accepted:
        opts['eval_strategy'] = 'epoch'
    elif 'evaluation_strategy' in accepted:
        opts['evaluation_strategy'] = 'epoch'
    else:
        raise RuntimeError('Transformers does not expose an evaluation strategy')
    if 'warmup_ratio' in accepted:
        opts['warmup_ratio'] = 0.05
    elif 'warmup_steps' in accepted:
        opts['warmup_steps'] = 100
    unsupported = sorted(set(opts)-set(accepted))
    if unsupported:
        raise TypeError(f'Unsupported training arguments: {unsupported}')
    return Seq2SeqTrainingArguments(**opts)


def _plot_loss(hist, name):
    fig, ax = plt.subplots(figsize=(8, 5))
    for key, legend in [('loss','Training loss (logged)'), ('eval_loss','Validation loss')]:
        if key in hist.columns:
            frame = hist.dropna(subset=['epoch', key])
            if not frame.empty:
                ax.plot(frame['epoch'], frame[key], marker='o', label=legend)
    ax.set(xlabel='Epoch', ylabel='Loss', title=f'{name}: training and validation loss')
    ax.grid(alpha=.25)
    if ax.lines: ax.legend()
    fig.tight_layout()
    fig.savefig(OUT/'figures'/f'{name}_learning_curve.png', dpi=300, bbox_inches='tight')
    plt.close(fig)


def _generate(frame, model, tokenizer, batch_size):
    outputs=[]
    model.eval()
    for start in range(0, len(frame), batch_size):
        batch=frame['dialect_sentence'].iloc[start:start+batch_size].astype(str).tolist()
        x=tokenizer(batch, return_tensors='pt', padding=True,
                    truncation=True, max_length=GEN_MAX_INPUT)
        x={k:v.to(model.device) for k,v in x.items()}
        with torch.inference_mode():
            tokens=model.generate(**x, max_new_tokens=GEN_MAX_TARGET,
                                  num_beams=GENERATOR_BEAMS)
        outputs.extend(tokenizer.batch_decode(tokens, skip_special_tokens=True))
    return outputs

if RUN_GENERATORS:
    assert torch.cuda.is_available(), 'Enable GPU in Colab before training generators.'
    name = GENERATOR_TO_RUN
    backbone = GEN_MODELS[name]
    print('TRAIN GENERATOR:', name, '| Transformers:', transformers.__version__)
    # Force the genuine SentencePiece tokenizer for BanglaT5.
    # A .model SentencePiece binary is NOT a TikToken BPE text file.
    if name == 'BanglaT5':
        import google.protobuf, sentencepiece
        print('Protobuf:', google.protobuf.__version__,
              '| SentencePiece:', sentencepiece.__version__)
        try:
            tokenizer = AutoTokenizer.from_pretrained(backbone, use_fast=False)
        except Exception as e:
            raise RuntimeError(
                'BanglaT5 slow SentencePiece tokenizer failed. Run the first '
                'installation cell, restart Colab runtime, and rerun from the '
                'top. Do not use TikToken fallback for spiece.model.'
            ) from e
    else:
        tokenizer = AutoTokenizer.from_pretrained(backbone, use_fast=False)
    print('Tokenizer:', type(tokenizer).__name__)
    assert tokenizer.pad_token_id is not None, 'Tokenizer has no pad token'
    assert len(tokenizer('আমি আজ স্কুলে যাচ্ছি।')['input_ids']) > 0
    config = AutoConfig.from_pretrained(backbone)
    if name == 'BanglaT5':
        config.tie_word_embeddings = False  # checkpoint contains different shared/head weights
    if name == 'mBART50':
        tokenizer.src_lang = 'bn_IN'
        tokenizer.tgt_lang = 'bn_IN'
        codes = getattr(tokenizer, 'lang_code_to_id', {})
        if 'bn_IN' not in codes:
            raise ValueError('Tokenizer does not expose bn_IN; inspect model/tokenizer version.')
        tokenizer.src_lang = 'bn_IN'
        tokenizer.tgt_lang = 'bn_IN'
    model = AutoModelForSeq2SeqLM.from_pretrained(backbone, config=config)
    if name == 'mBART50':
        model.generation_config.forced_bos_token_id = tokenizer.lang_code_to_id['bn_IN']
        model.config.forced_bos_token_id = tokenizer.lang_code_to_id['bn_IN']
    train_ds = _make_seq_dataset(splits['train'], tokenizer)
    val_ds = _make_seq_dataset(splits['val'], tokenizer)
    collator = DataCollatorForSeq2Seq(tokenizer, model=model, label_pad_token_id=-100)
    sample = collator([train_ds[i] for i in range(min(2, len(train_ds)))])
    assert (sample['labels'] != -100).any(), 'All target labels masked'
    model.eval()
    with torch.no_grad():
        initial_loss = model(**sample).loss
    assert torch.isfinite(initial_loss), f'Initial loss is not finite: {initial_loss}'
    print('Initial FP32 loss:', float(initial_loss))
    args = _make_args(name)
    trainer = Seq2SeqTrainer(model=model, args=args, train_dataset=train_ds,
        eval_dataset=val_ds, processing_class=tokenizer, data_collator=collator)
    started = time.perf_counter()
    if torch.cuda.is_available():torch.cuda.reset_peak_memory_stats()
    trainer.train()
    _peak_mb=torch.cuda.max_memory_allocated()/1024**2 if torch.cuda.is_available() else np.nan
    complexity_records.append(dict(task='standardization',family='Transformer',model=name,device='GPU' if torch.cuda.is_available() else 'CPU',train_seconds=time.perf_counter()-started,inference_ms_per_sample=np.nan,parameters=sum(p.numel() for p in trainer.model.parameters()),gpu_peak_mb=_peak_mb))
    elapsed = time.perf_counter() - started
    hist = pd.DataFrame(trainer.state.log_history)
    hist.to_csv(OUT/'logs'/f'{name}_history.csv', index=False)
    val_losses = hist['eval_loss'].dropna().to_numpy(dtype=float) if 'eval_loss' in hist else np.array([])
    if len(val_losses) == 0 or not np.isfinite(val_losses).all():
        raise RuntimeError(f'{name}: non-finite or missing validation loss; inspect saved logs before evaluation.')
    if 'loss' in hist:
        train_losses = hist['loss'].dropna().to_numpy(dtype=float)
        if len(train_losses) and not np.isfinite(train_losses).all():
            raise RuntimeError(f'{name}: non-finite training loss; inspect logs.')
    best_dir = OUT/'models'/f'{name}_best'
    best_dir.mkdir(parents=True, exist_ok=True)
    trainer.save_model(str(best_dir))  # best validation checkpoint, due to load_best_model_at_end
    tokenizer.save_pretrained(str(best_dir))
    (OUT/'logs'/f'{name}_run.json').write_text(json.dumps({
        'model': backbone, 'train_rows': len(train_ds), 'val_rows': len(val_ds),
        'best_checkpoint': trainer.state.best_model_checkpoint,
        'best_eval_loss': float(min(val_losses)), 'train_seconds': elapsed,
        'gradient_accumulation_steps': GENERATOR_GRAD_ACCUM,
        'fp16': False, 'bf16': False,
    }, indent=2), encoding='utf-8')
    _plot_loss(hist, name)
    model = trainer.model
    for part, frame in [('validation', splits['val']), ('internal_test', splits['test'])] + list(external_clean.items()):
        if frame.empty: continue
        print('EVALUATING', name, part, len(frame))
        predictions = _generate(frame, model, tokenizer, GENERATOR_EVAL_BATCH)
        eval_generator(name, part, frame, predictions)  # existing BLEU/chrF/EM evaluator
    print('Saved:', best_dir, 'and 300 DPI learning curve')
    del trainer, model, tokenizer, train_ds, val_ds, collator
    gc.collect()
    torch.cuda.empty_cache()


TRAIN GENERATOR: BanglaT5 | Transformers: 4.51.3
Protobuf: 6.33.6 | SentencePiece: 0.2.1
Tokenizer: T5Tokenizer


pytorch_model.bin:   0%|          | 0.00/990M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

Map:   0%|          | 0/13036 [00:00<?, ? examples/s]

Map:   0%|          | 0/2806 [00:00<?, ? examples/s]

Passing a tuple of `past_key_values` is deprecated and will be removed in Transformers v4.48.0. You should pass an instance of `EncoderDecoderCache` instead, e.g. `past_key_values=EncoderDecoderCache.from_legacy_cache(past_key_values)`.


Initial FP32 loss: 12.099390029907227


Epoch,Training Loss,Validation Loss
1,6.567800,2.367064
2,3.116800,1.970014
3,2.737500,1.859472


There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight'].


EVALUATING BanglaT5 validation 2806
EVALUATING BanglaT5 internal_test 2808
EVALUATING BanglaT5 External1 1922
EVALUATING BanglaT5 External2 2845
EVALUATING BanglaT5 External3 13996
Saved: /content/drive/MyDrive/BANGLA DIALECT PAPER/DATASET/reviewer_ready_v8_reviewer_complete/models/BanglaT5_best and 300 DPI learning curve


Optional word-level external evaluation (reported separately; no mixing with clauses)


Code cell 35

In [ ]:
if RUN_EXTERNAL_WORD_LEVEL:
    words=pd.read_csv(EXT_ROOT/'external2'/'BD-Dialect_Words.csv')
    word_raw=wide_to_long(words,'Standard Bangla Languge',
        {'Nowakhali Language':'Noakhali','Sylheti Language':'Sylhet','Chittagong Language':'Chittagong'},'External2_Words')
    mi,si=near_hit_mask(word_raw.dialect_sentence.tolist(),internal_inputs)
    mr,sr=near_hit_mask(word_raw.standard_bengali.tolist(),internal_refs)
    word_clean=word_raw.loc[~(mi|mr)].reset_index(drop=True)
    word_clean.to_csv(OUT/'prepared_external'/'External2_Words_strict_disjoint.csv',index=False)
    print('Word-level eligible rows:',len(word_clean))
    print('Evaluate saved models on word_clean separately; do not mix with sentence-level scores.')


14. Comparison curves — PNG only (300 DPI, deterministic filenames)


Code cell 37

In [ ]:
results=pd.read_csv(OUT/'metrics'/'all_results.csv')
for task,metric in [('classification','macro_f1'),('classification','accuracy'),('classification','macro_ovr_auc'),('standardization','bleu'),('standardization','chrf')]:
    z=results[(results.task==task)&results[metric].notna()].drop_duplicates(['model','dataset'],keep='last').copy() if metric in results else pd.DataFrame()
    if z.empty:continue
    fig,ax=plt.subplots(figsize=(max(9,len(z.model.unique())*1.4),6))
    for ds,part in z.groupby('dataset'):
        ax.plot(part.model,part[metric],marker='o',label=ds)
    ax.set(xlabel='Model',ylabel=metric,title=f'{task}: {metric} across datasets');ax.tick_params(axis='x',rotation=45)
    ax.legend(loc='best',fontsize=8);savefig(f'comparison_{task}_{metric}')
# Summary tables
for task in results.task.dropna().unique():
    results[results.task==task].to_csv(OUT/'metrics'/f'{task}_comparison_table.csv',index=False)
print('Saved plots:',len(list((OUT/'figures').glob('*.png'))),'PNG files')
display(results)


Saved plots: 100 PNG files


,task,model,dataset,n,accuracy,macro_f1,covered_class_macro_f1,gold_classes_present,weighted_f1,macro_precision,...,macro_ovr_auc,auc_evaluable_classes,auc_definition,bleu,chrf,exact_match,rouge_l,empty_prediction_rate,mean_prediction_tokens,mean_reference_tokens
0,classification,BanglaBERT,External1,1922,0.607700,0.261518,0.653796,"Barisal,Khulna",0.659325,0.324286,...,0.815229,2.0,mean valid one-vs-rest class AUCs; not full 5-...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,classification,BanglaBERT,External2,2845,0.348330,0.234529,0.390881,"Chittagong,Noakhali,Sylhet",0.390933,0.301217,...,0.650522,3.0,mean valid one-vs-rest class AUCs; not full 5-...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,classification,BanglaBERT,External3,13996,0.679837,0.463073,0.578841,"Barisal,Chittagong,Noakhali,Sylhet",0.703844,0.461857,...,0.870004,4.0,mean valid one-vs-rest class AUCs; not full 5-...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,classification,BanglaBERT,internal_test,2808,0.916667,0.916778,0.916778,"Barisal,Chittagong,Khulna,Noakhali,Sylhet",0.916632,0.916853,...,0.990825,5.0,mean valid one-vs-rest class AUCs; not full 5-...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,classification,BanglaBERT,validation,2806,0.921240,0.921591,0.921591,"Barisal,Chittagong,Khulna,Noakhali,Sylhet",0.921402,0.922054,...,0.990567,5.0,mean valid one-vs-rest class AUCs; not full 5-...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,classification,LinearSVM,External1,1922,0.661811,0.276987,0.692468,"Barisal,Khulna",0.697759,0.327795,...,0.828811,2.0,mean valid one-vs-rest class AUCs; not full 5-...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,classification,LinearSVM,External2,2845,0.429877,0.289081,0.481802,"Chittagong,Noakhali,Sylhet",0.481836,0.338848,...,0.698044,3.0,mean valid one-vs-rest class AUCs; not full 5-...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,classification,LinearSVM,External3,13996,0.749786,0.518368,0.647959,"Barisal,Chittagong,Noakhali,Sylhet",0.778279,0.529959,...,0.918392,4.0,mean valid one-vs-rest class AUCs; not full 5-...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,classification,LinearSVM,internal_test,2808,0.942308,0.942409,0.942409,"Barisal,Chittagong,Khulna,Noakhali,Sylhet",0.942346,0.942617,...,0.995428,5.0,mean valid one-vs-rest class AUCs; not full 5-...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,classification,LinearSVM,validation,2806,0.940485,0.940565,0.940565,"Barisal,Chittagong,Khulna,Noakhali,Sylhet",0.940481,0.940830,...,0.995892,5.0,mean valid one-vs-rest class AUCs; not full 5-...,NaN,NaN,NaN,NaN,NaN,NaN,NaN


15. Final integrity assertions and manifest
    Claims permitted: exact cross-split disjointness, near-duplicate threshold-based grouping, exact and near-threshold external overlap exclusion. Claims NOT supported: all semantic paraphrases, same-speaker or same-source independence without metadata. If audit thresholds or preprocessing are changed, retrain from the beginning.
    Code cell 38
    Cell 38 of 39


Code cell 39

In [ ]:
for a,b in [('train','val'),('train','test'),('val','test')]:
    for col in ['group_id','dialect_sentence','standard_bengali']:
        assert set(splits[a][col]).isdisjoint(splits[b][col]),f'Internal leakage {a}-{b}:{col}'
for name,z in external_clean.items():
    assert set(z.dialect_sentence).isdisjoint(internal_inputs),f'External input exact leakage {name}'
    assert set(z.standard_bengali).isdisjoint(internal_refs),f'External reference exact leakage {name}'
manifest={'seed':SEED,'source_internal':str(INTERNAL_FILE),'external_root':str(EXT_ROOT),
          'threshold_char_tfidf_cosine':NEAR_THRESHOLD,'near_ngram':NEAR_NGRAM,
          'split_sizes':{k:len(v) for k,v in splits.items()},
          'external_sizes':{k:len(v) for k,v in external_clean.items()},
          'models_requested':{'classical':list(classical),'transformers':CLS_MODELS,'generators':GEN_MODELS,'lstm':RUN_LSTM},
          'python_torch':torch.__version__,'limitations':'No speaker/source IDs; similarity threshold does not prove semantic independence.'}
(OUT/'audit'/'experiment_manifest.json').write_text(json.dumps(manifest,indent=2,ensure_ascii=False),encoding='utf-8')
print('COMPLETED. All saved to:',OUT)
print('Figures 300 DPI:',OUT/'figures')


COMPLETED. All saved to: /content/drive/MyDrive/BANGLA DIALECT PAPER/DATASET/reviewer_ready_v8_reviewer_complete
Figures 300 DPI: /content/drive/MyDrive/BANGLA DIALECT PAPER/DATASET/reviewer_ready_v8_reviewer_complete/figures


16. Reviewer-focused computational complexity and performance comparison
    This cell uses measured training time and memory, and reports parameter counts for neural networks. It does not claim theoretical FLOPs. Classical ML model parameter counts are not directly comparable with neural network parameters. CPU and GPU execution times must not be interpreted as hardware-normalized speed comparisons. Rerun experiments on the same hardware for fair benchmarking.


Code cell 41

In [ ]:
from pathlib import Path
import os, platform, psutil
from sklearn.metrics import f1_score

# Recover LSTM measurements saved by its training cell.
lstm_file=OUT/'metrics'/'LSTM_compute.json'
if lstm_file.exists():
    v=json.loads(lstm_file.read_text())
    complexity_records.append({'task':'standardization','family':'LSTM','model':'LSTM','device':v['device'],'train_seconds':v['train_seconds'],'inference_ms_per_sample':np.nan,'parameters':v['parameters'],'gpu_peak_mb':np.nan})

# Recover previously completed model runs across separate Colab sessions.
for logfile in sorted((OUT/'logs').glob('*_run.json')):
    try:
        rec=json.loads(logfile.read_text())
        model_name=logfile.name.removesuffix('_run.json')
        if not any(r.get('model')==model_name for r in complexity_records):
            complexity_records.append(dict(task='standardization',family='Transformer',model=model_name,
                device='GPU (recorded)',train_seconds=rec.get('train_seconds',np.nan),
                inference_ms_per_sample=np.nan,parameters=np.nan,gpu_peak_mb=np.nan))
    except (ValueError,KeyError) as exc:
        print('Skipping malformed run log:',logfile.name,exc)
compute=pd.DataFrame(complexity_records)
if not compute.empty:
    compute=compute.drop_duplicates(['task','model'],keep='last')
# Disk footprint from saved model artifacts (actual storage, not peak RAM).
def model_size_mb(name):
    candidates=[OUT/'models'/f'{name}.joblib',OUT/'models'/f'{name}_best',OUT/'models'/name,OUT/'models'/'LSTM_best.pt']
    path=next((p for p in candidates if p.exists() and (name!='LSTM' or p.name=='LSTM_best.pt')),None)
    if path is None:return np.nan
    paths=[path] if path.is_file() else list(path.rglob('*'))
    return sum(p.stat().st_size for p in paths if p.is_file())/1024**2
if not compute.empty:
    compute['disk_size_mb']=compute.model.map(model_size_mb)
    compute.to_csv(OUT/'metrics'/'computational_complexity.csv',index=False)
    print(compute.to_string(index=False))

res_path=OUT/'metrics'/'all_results.csv'
if res_path.exists() and not compute.empty:
    res=pd.read_csv(res_path)
    joined=res.merge(compute,on=['task','model'],how='left',validate='many_to_one')
    joined.to_csv(OUT/'metrics'/'performance_vs_compute.csv',index=False)
    for task,metric in [('classification','macro_f1'),('standardization','chrf')]:
        z=joined[(joined.task==task)&(joined.dataset=='internal_test')].dropna(subset=['train_seconds',metric]) if metric in joined else pd.DataFrame()
        if z.empty:continue
        fig,ax=plt.subplots(figsize=(8,5));ax.scatter(z.train_seconds/60,z[metric]);
        for _,r in z.iterrows():ax.annotate(r.model,(r.train_seconds/60,r[metric]),xytext=(4,4),textcoords='offset points')
        ax.set(xlabel='Measured training time (minutes; hardware-dependent)',ylabel=metric,title=f'{task}: quality versus training cost');savefig(f'{task}_quality_vs_train_time')

hardware={'python':platform.python_version(),'torch':torch.__version__,'gpu':torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,'cpu':platform.processor(),'cpu_count':os.cpu_count(),'ram_gb':round(psutil.virtual_memory().total/1024**3,2),'seed':SEED,'split_target':[TRAIN_FRAC,VAL_FRAC,TEST_FRAC]}
(OUT/'audit'/'hardware.json').write_text(json.dumps(hardware,indent=2),encoding='utf-8')
print('Saved reviewer complexity tables and hardware manifest.')


           task       family              model device  train_seconds  inference_ms_per_sample  parameters  gpu_peak_mb  disk_size_mb
 classification Classical ML LogisticRegression    CPU      18.635544                 0.172191         NaN          NaN      6.249180
 classification Classical ML          LinearSVM    CPU       2.033739                 0.090174         NaN          NaN      6.249054
 classification Classical ML      MultinomialNB    CPU       1.474203                 0.077208         NaN          NaN      9.304772
 classification Classical ML       RandomForest    CPU      96.867648                 0.250372         NaN          NaN    234.717465
 classification  Transformer         BanglaBERT    GPU     773.967530                      NaN 164400389.0  3163.865234    632.985594
 classification  Transformer              mBERT    GPU     868.032071                      NaN 177857285.0  3420.829102    682.235657
 classification  Transformer              MuRIL    GPU     988

17. Final results audit — no duplicated learning curves, confusion matrices or ROC curves


Code cell 43

In [ ]:
# FINAL REVIEW / REPRODUCIBILITY AUDIT — no duplicate confusion matrices or ROC figures.
# Existing classifier evaluation already writes one deterministic CM and ROC filename per model/dataset.
# Existing classifier and generator cells already write one learning curve per model.
from pathlib import Path
import pandas as pd, numpy as np, json
results_path=OUT/'metrics'/'all_results.csv'
assert results_path.exists(), 'Run the model evaluation sections first.'
r=pd.read_csv(results_path)
assert not r.duplicated(['task','model','dataset']).any(), 'Duplicate metric keys detected.'
print('Unique result rows:',len(r))
print('Completed models by task:')
print(r.groupby('task')['model'].unique().to_string())
print('Internal test results:')
show=['task','model','dataset','n','accuracy','macro_f1','macro_ovr_auc','bleu','chrf','rouge_l','exact_match','empty_prediction_rate']
display(r[r.dataset.eq('internal_test')][[x for x in show if x in r.columns]].sort_values(['task','model']))
fig_dir=OUT/'figures'
figure_files=list(fig_dir.glob('*.png'))
print('Unique PNG filenames:',len({f.name for f in figure_files}))
print('Confusion matrices:',sum(f.name.endswith('_confusion_matrix.png') for f in figure_files))
print('ROC curves:',sum(f.name.endswith('_roc.png') for f in figure_files))
print('Learning curves:',sum(f.name.endswith('_learning_curve.png') for f in figure_files))
# Check standardization outputs without fabricating missing results.
for name in ['LSTM','BanglaT5','mT5','mBART50']:
    f=OUT/'predictions'/f'{name}_internal_test_generation.csv'
    if f.exists():
        d=pd.read_csv(f).fillna('')
        if {'reference','prediction'}.issubset(d.columns):
            print(name,'rows:',len(d),'empty:',int(d.prediction.astype(str).str.strip().eq('').sum()),
                  'average prediction tokens:',round(d.prediction.astype(str).str.split().str.len().mean(),2))
    else:
        print(name,': no internal-test generation file (not evaluated)')
report={'dataset_clean_path':str(OUT/'audit'/'final_clean_dataset.csv'),
        'split_path':str(OUT/'splits'),'results_rows':len(r),
        'unique_png_figures':len({f.name for f in figure_files}),
        'notes':['ROC AUC averages only evaluable classes in partial-label external sets',
                 'CPU/GPU timing is hardware-dependent, not theoretical FLOPs',
                 'Missing model results are not imputed']}
(OUT/'audit'/'final_review_manifest.json').write_text(json.dumps(report,indent=2),encoding='utf-8')
print('Final review manifest saved.')


Unique result rows: 50
Completed models by task:
task
classification     [BanglaBERT, LinearSVM, LogisticRegression, Mu...
standardization                                     [BanglaT5, LSTM]
Internal test results:


,task,model,dataset,n,accuracy,macro_f1,macro_ovr_auc,bleu,chrf,rouge_l,exact_match,empty_prediction_rate
3,classification,BanglaBERT,internal_test,2808,0.916667,0.916778,0.990825,NaN,NaN,NaN,NaN,NaN
8,classification,LinearSVM,internal_test,2808,0.942308,0.942409,0.995428,NaN,NaN,NaN,NaN,NaN
13,classification,LogisticRegression,internal_test,2808,0.926638,0.926753,0.993847,NaN,NaN,NaN,NaN,NaN
18,classification,MuRIL,internal_test,2808,0.924145,0.924178,0.988206,NaN,NaN,NaN,NaN,NaN
23,classification,MultinomialNB,internal_test,2808,0.919872,0.919953,0.993667,NaN,NaN,NaN,NaN,NaN
28,classification,RandomForest,internal_test,2808,0.883191,0.883046,0.984885,NaN,NaN,NaN,NaN,NaN
33,classification,XLM-R,internal_test,2808,0.918447,0.918600,0.990577,NaN,NaN,NaN,NaN,NaN
38,classification,mBERT,internal_test,2808,0.924501,0.924652,0.992027,NaN,NaN,NaN,NaN,NaN
43,standardization,BanglaT5,internal_test,2808,NaN,NaN,NaN,34.523900,60.295243,0.0,0.102920,0.0
48,standardization,LSTM,internal_test,2808,NaN,NaN,NaN,6.197999,23.113532,0.0,0.007835,0.0


Unique PNG filenames: 102
Confusion matrices: 40
ROC curves: 40
Learning curves: 10
LSTM rows: 2808 empty: 0 average prediction tokens: 6.05
BanglaT5 rows: 2808 empty: 0 average prediction tokens: 6.3
mT5 : no internal-test generation file (not evaluated)
mBART50 : no internal-test generation file (not evaluated)
Final review manifest saved.


18. Reviewer audit: split integrity, distribution, and statistical uncertainty
Run after dataset preparation and model evaluation. These are additional audits, not repeated training or duplicate figures.


Code cell 45

In [ ]:
from sklearn.metrics import f1_score, accuracy_score
from sklearn.model_selection import train_test_split
from scipy.stats import bootstrap
import hashlib

AUDIT_DIR = OUT / 'audit'
AUDIT_DIR.mkdir(parents=True,exist_ok=True)
# Verify actual group disjointness and reference disjointness, not just input overlap.
for a,b in [('train','val'),('train','test'),('val','test')]:
    for key in ('dialect_sentence','standard_bengali'):
        overlap=set(splits[a][key].astype(str)) & set(splits[b][key].astype(str))
        print(f'{a}/{b} {key} exact overlap: {len(overlap)}')
        assert not overlap, f'Leakage detected for {key}: {a} vs {b}'
rows=[]
for name,part in splits.items():
    for dialect in sorted(set().union(*(set(x.region) for x in splits.values()))):
        rows.append({'split':name,'region':dialect,'n':int((part.region==dialect).sum()),'split_total':len(part)})
pd.DataFrame(rows).to_csv(AUDIT_DIR/'split_class_distribution.csv',index=False)
print('Actual split fractions:',{k:round(len(v)/sum(map(len,splits.values())),4) for k,v in splits.items()})
# Reproducible data fingerprints allow reviewer to verify the exact partitions.
manifest={}
for name,part in splits.items():
    canonical='\n'.join(sorted(part[['region','dialect_sentence','standard_bengali']].astype(str).agg('\t'.join,axis=1)))
    manifest[name]={'rows':len(part),'sha256':hashlib.sha256(canonical.encode('utf-8')).hexdigest()}
(AUDIT_DIR/'split_manifest_sha256.json').write_text(json.dumps(manifest,indent=2),encoding='utf-8')
print('Saved split audit and fingerprints')


train/val dialect_sentence exact overlap: 0
train/val standard_bengali exact overlap: 0
train/test dialect_sentence exact overlap: 0
train/test standard_bengali exact overlap: 0
val/test dialect_sentence exact overlap: 0
val/test standard_bengali exact overlap: 0
Actual split fractions: {'train': 0.699, 'val': 0.1505, 'test': 0.1506}
Saved split audit and fingerprints


19. Reviewer audit: classification confidence intervals and error breakdown
Uses saved predictions; does not retrain. A bootstrap interval is descriptive and assumes independent test rows. For correlated source groups, use group-level bootstrap instead.


Code cell 47

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
from numpy.random import default_rng
PRED_DIR=OUT/'predictions'
print('Prediction files:',len(list(PRED_DIR.glob('*.csv'))))
ci_rows=[]
for path in sorted(PRED_DIR.glob('*.csv')):
    if 'generation' in path.name: continue
    try: z=pd.read_csv(path)
    except Exception: continue
    # Only evaluate files with explicit true/predicted class columns.
    truth=next((x for x in ('true_label','true_region','region','y_true') if x in z),None)
    pred=next((x for x in ('predicted_label','predicted_region','prediction','y_pred') if x in z),None)
    if truth is None or pred is None or len(z)<2: continue
    yt=z[truth].astype(str).to_numpy(); yp=z[pred].astype(str).to_numpy()
    # Convert numeric class IDs if present.
    if all(v.isdigit() for v in set(yt)|set(yp)) and 'I2L' in globals():
        yt=np.array([I2L.get(int(x),x) for x in yt]);yp=np.array([I2L.get(int(x),x) for x in yp])
    rng=default_rng(42); values=[]
    for _ in range(1000):
        ids=rng.integers(0,len(yt),size=len(yt))
        values.append(f1_score(yt[ids],yp[ids],labels=LABELS,average='macro',zero_division=0))
    lo,hi=np.quantile(values,[.025,.975])
    ci_rows.append({'file':path.name,'n':len(z),'macro_f1':f1_score(yt,yp,labels=LABELS,average='macro',zero_division=0),'ci95_low':lo,'ci95_high':hi})
if ci_rows:
    ci=pd.DataFrame(ci_rows)
    ci.to_csv(OUT/'metrics'/'classification_bootstrap_ci.csv',index=False)
    display(ci)
else:
    print('No compatible saved classification prediction files. Check column names before calculating confidence intervals.')


Prediction files: 50
No compatible saved classification prediction files. Check column names before calculating confidence intervals.


20. Reviewer audit: generation errors and length-stratified evaluation
Checks hallucination-like repetitions, empty outputs, length bias and example pairs. No duplicate learning curves, confusion matrices or ROC figures.


Code cell 49

In [ ]:
import sacrebleu
error_rows=[];bucket_rows=[]
for path in sorted((OUT/'predictions').glob('*_generation.csv')):
    z=pd.read_csv(path).fillna('')
    if not {'input','reference','prediction'}.issubset(z.columns): continue
    z['ref_tokens']=z.reference.astype(str).str.split().str.len()
    z['pred_tokens']=z.prediction.astype(str).str.split().str.len()
    z['exact_match']=z.reference.astype(str).str.strip().eq(z.prediction.astype(str).str.strip())
    z['repetition_flag']=z.prediction.astype(str).map(lambda s: len(s.split())>=4 and len(set(s.split()))<=max(1,len(s.split())//3))
    z['length_ratio']=z.pred_tokens/(z.ref_tokens.clip(lower=1))
    z['length_bucket']=pd.cut(z.ref_tokens,bins=[-1,5,10,20,10**9],labels=['1-5','6-10','11-20','21+'])
    error_rows.append({'file':path.name,'n':len(z),'empty_rate':float(z.pred_tokens.eq(0).mean()),'repetition_rate':float(z.repetition_flag.mean()),'exact_match':float(z.exact_match.mean()),'mean_length_ratio':float(z.length_ratio.mean())})
    for bucket,group in z.groupby('length_bucket',observed=True):
        bucket_rows.append({'file':path.name,'reference_length':str(bucket),'n':len(group),'bleu':sacrebleu.corpus_bleu(group.prediction.astype(str).tolist(),[group.reference.astype(str).tolist()]).score,'chrf':sacrebleu.corpus_chrf(group.prediction.astype(str).tolist(),[group.reference.astype(str).tolist()]).score})
    z.sort_values(['exact_match','length_ratio'],ascending=[True,False]).head(50).to_csv(OUT/'metrics'/f'{path.stem}_error_examples.csv',index=False)
if error_rows:
    pd.DataFrame(error_rows).to_csv(OUT/'metrics'/'generation_error_audit.csv',index=False)
    pd.DataFrame(bucket_rows).to_csv(OUT/'metrics'/'generation_length_stratified.csv',index=False)
    display(pd.DataFrame(error_rows))
else: print('No completed generator prediction files yet.')


,file,n,empty_rate,repetition_rate,exact_match,mean_length_ratio
0,BanglaT5_External1_generation.csv,1922,0.0,0.0,0.002601,1.118732
1,BanglaT5_External2_generation.csv,2845,0.0,0.0,0.000351,1.797422
2,BanglaT5_External3_generation.csv,13996,0.0,0.0,0.013718,0.998960
3,BanglaT5_internal_test_generation.csv,2808,0.0,0.0,0.102920,1.032307
4,BanglaT5_validation_generation.csv,2806,0.0,0.0,0.091233,1.043277
5,LSTM_External1_generation.csv,1922,0.0,0.0,0.000000,1.105328
6,LSTM_External2_generation.csv,2845,0.0,0.0,0.000000,1.475688
7,LSTM_External3_generation.csv,13996,0.0,0.0,0.002572,0.973919
8,LSTM_internal_test_generation.csv,2808,0.0,0.0,0.007835,0.996555
9,LSTM_validation_generation.csv,2806,0.0,0.0,0.002495,0.995856


21. Reviewer audit: hardware fairness, missing computational measurements, and result completeness
CPU and GPU timings are retained with device labels and are not presented as hardware-normalized rankings.


Code cell 51

In [ ]:
metrics_dir=OUT/'metrics'
compute_path=metrics_dir/'computational_complexity.csv'
results_path=metrics_dir/'all_results.csv'
if compute_path.exists():
    compute=pd.read_csv(compute_path)
    needed=['model','device','train_seconds','inference_ms_per_sample','parameters','gpu_peak_mb','disk_size_mb']
    print('Computational metrics completeness:')
    display(compute[[c for c in needed if c in compute.columns]])
    pd.DataFrame({'metric':needed,'missing':[int(compute[c].isna().sum()) if c in compute else len(compute) for c in needed]}).to_csv(metrics_dir/'compute_missingness_audit.csv',index=False)
else: print('Run computational complexity cell first.')
if results_path.exists():
    r=pd.read_csv(results_path)
    expected_cls=['LogisticRegression','LinearSVM','MultinomialNB','RandomForest','BanglaBERT','mBERT','MuRIL','XLM-R']
    expected_gen=['LSTM','BanglaT5','mT5','mBART50']
    for task,models in [('classification',expected_cls),('standardization',expected_gen)]:
        have=set(r.loc[(r.task==task)&(r.dataset=='internal_test'),'model'])
        print(task,'internal-test missing:',sorted(set(models)-have))
    assert not r.duplicated(['task','model','dataset']).any(),'Duplicate results in all_results.csv'
    print('Result keys unique.')
# Explicit PNG-only export audit (existing PDFs are not removed).
figs=OUT/'figures'
print('PNG figures:',len(list(figs.glob('*.png'))))
print('Existing PDF files (not generated by this section):',len(list(figs.glob('*.pdf'))))


Computational metrics completeness:


,model,device,train_seconds,inference_ms_per_sample,parameters,gpu_peak_mb,disk_size_mb
0,LogisticRegression,CPU,18.635544,0.172191,NaN,NaN,6.249180
1,LinearSVM,CPU,2.033739,0.090174,NaN,NaN,6.249054
2,MultinomialNB,CPU,1.474203,0.077208,NaN,NaN,9.304772
3,RandomForest,CPU,96.867648,0.250372,NaN,NaN,234.717465
4,BanglaBERT,GPU,773.967530,NaN,164400389.0,3163.865234,632.985594
5,mBERT,GPU,868.032071,NaN,177857285.0,3420.829102,682.235657
6,MuRIL,GPU,988.616055,NaN,237560069.0,4560.170898,915.378068
7,XLM-R,GPU,1120.920732,NaN,278047493.0,5330.135742,1081.823714
8,BanglaT5,GPU,4761.343632,NaN,247577856.0,4896.683594,945.557880
9,LSTM,cuda,44.765726,NaN,8368898.0,NaN,31.928327


classification internal-test missing: []
standardization internal-test missing: ['mBART50', 'mT5']
Result keys unique.
PNG figures: 102
Existing PDF files (not generated by this section): 0


22. Reviewer-ready paired significance tests (McNemar + paired bootstrap)
Use identical internal-test examples and predictions; do not compare unmatched rows. Results are deterministic and saved by replacement, not appended.


Code cell 53

In [ ]:
from scipy.stats import binomtest
from itertools import combinations
from sklearn.metrics import f1_score
from numpy.random import default_rng

PRED_DIR = OUT / 'predictions'
TEST_ORDER = splits['test'].reset_index(drop=True)
TEST_GOLD = TEST_ORDER['region'].astype(str).to_numpy()
TEST_INPUT = TEST_ORDER['dialect_sentence'].astype(str).to_numpy()

def load_aligned_classification(name):
    path = PRED_DIR / f'{name}_internal_test.csv'
    if not path.exists():
        return None
    d = pd.read_csv(path).fillna('')
    if not {'true','predicted'}.issubset(d.columns) or len(d) != len(TEST_GOLD):
        print('Skip unaligned prediction file:', path.name)
        return None
    # Current notebook writes predictions in original test-set order.
    if not np.array_equal(d['true'].astype(str).to_numpy(), TEST_GOLD):
        print('Skip mismatched test label order:', path.name)
        return None
    return d['predicted'].astype(str).to_numpy()

models = ['LogisticRegression','LinearSVM','MultinomialNB','RandomForest',
          'BanglaBERT','mBERT','MuRIL','XLM-R']
preds = {name: load_aligned_classification(name) for name in models}
preds = {name: value for name, value in preds.items() if value is not None}
paired_rows=[]
for a,b in combinations(sorted(preds),2):
    pa,pb=preds[a],preds[b]
    a_ok=pa==TEST_GOLD; b_ok=pb==TEST_GOLD
    a_only=int(np.sum(a_ok & ~b_ok));b_only=int(np.sum(~a_ok & b_ok))
    p=float(binomtest(min(a_only,b_only),n=a_only+b_only,p=.5).pvalue) if a_only+b_only else 1.0
    rng=default_rng(SEED)
    diffs=[]
    for _ in range(1000):
        idx=rng.integers(0,len(TEST_GOLD),len(TEST_GOLD))
        diffs.append(f1_score(TEST_GOLD[idx],pa[idx],labels=LABELS,average='macro',zero_division=0)-
                     f1_score(TEST_GOLD[idx],pb[idx],labels=LABELS,average='macro',zero_division=0))
    lo,hi=np.quantile(diffs,[.025,.975])
    paired_rows.append({'model_a':a,'model_b':b,'n':len(TEST_GOLD),
                        'a_only_correct':a_only,'b_only_correct':b_only,
                        'mcnemar_exact_p':p,'macro_f1_diff_a_minus_b':float(np.mean(diffs)),
                        'diff_ci95_low':float(lo),'diff_ci95_high':float(hi)})
paired=pd.DataFrame(paired_rows)
if not paired.empty:
    # Holm family-wise correction over the comparisons performed.
    from statsmodels.stats.multitest import multipletests
    paired['holm_adjusted_p']=multipletests(paired.mcnemar_exact_p,method='holm')[1]
    paired.to_csv(OUT/'metrics'/'paired_classifier_significance.csv',index=False)
    display(paired.sort_values('holm_adjusted_p').head(20))
else:
    print('Need at least two aligned saved internal-test classification prediction files.')


,model_a,model_b,n,a_only_correct,b_only_correct,mcnemar_exact_p,macro_f1_diff_a_minus_b,diff_ci95_low,diff_ci95_high,holm_adjusted_p
10,LinearSVM,RandomForest,2808,205,39,2.210310e-28,0.059340,0.049365,0.069500,6.188869e-27
15,LogisticRegression,RandomForest,2808,166,44,7.270580e-18,0.043626,0.034631,0.053648,1.963057e-16
26,RandomForest,mBERT,2808,94,210,2.441592e-11,-0.041480,-0.053691,-0.030040,6.348139e-10
22,MultinomialNB,RandomForest,2808,175,72,4.511514e-11,0.036864,0.026904,0.047615,1.127878e-09
19,MuRIL,RandomForest,2808,214,99,7.184755e-11,0.041235,0.028983,0.054293,1.724341e-09
25,RandomForest,XLM-R,2808,108,207,2.601526e-08,-0.035623,-0.048258,-0.023819,5.983509e-07
9,LinearSVM,MultinomialNB,2808,102,39,1.104220e-07,0.022476,0.014302,0.030937,2.429285e-06
4,BanglaBERT,RandomForest,2808,206,112,1.503405e-07,0.033756,0.021660,0.046719,3.157151e-06
0,BanglaBERT,LinearSVM,2808,72,144,1.091992e-06,-0.025584,-0.035308,-0.014924,2.183985e-05
7,LinearSVM,LogisticRegression,2808,66,22,2.879391e-06,0.015714,0.009565,0.022484,5.470842e-05


23. Inference latency benchmark for saved classification models
Same internal test inputs, fixed batch sizes, warm-up, and GPU synchronization. CPU/GPU devices are recorded; do not interpret different devices as hardware-normalized speed.

In [ ]:
import time, gc, joblib
from pathlib import Path
from transformers import AutoTokenizer, AutoModelForSequenceClassification

BENCHMARK_BATCH = 16
BENCHMARK_MAX_LENGTH = 128
BENCHMARK_REPEATS = 3
BENCHMARK_SAMPLES = min(256, len(splits['test']))
bench_texts = splits['test'].dialect_sentence.astype(str).iloc[:BENCHMARK_SAMPLES].tolist()

def _sync():
    if torch.cuda.is_available(): torch.cuda.synchronize()

def bench_classical(name):
    f=OUT/'models'/f'{name}.joblib'
    if not f.exists():return None
    mdl=joblib.load(f)
    for _ in range(2):mdl.predict(bench_texts[:min(16,len(bench_texts))])
    timings=[]
    for _ in range(BENCHMARK_REPEATS):
        start=time.perf_counter();_ = mdl.predict(bench_texts)
        timings.append(1000*(time.perf_counter()-start)/len(bench_texts))
    return {'task':'classification','model':name,'device':'CPU',
            'inference_ms_per_sample':float(np.median(timings)),
            'benchmark_n':len(bench_texts),'batch_size':BENCHMARK_BATCH,'repeats':BENCHMARK_REPEATS}

def bench_transformer(name):
    folder=OUT/'models'/name
    if not folder.exists():return None
    device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    tok=AutoTokenizer.from_pretrained(str(folder))
    mdl=AutoModelForSequenceClassification.from_pretrained(str(folder)).to(device).eval()
    def run(texts):
        for start in range(0,len(texts),BENCHMARK_BATCH):
            batch=tok(texts[start:start+BENCHMARK_BATCH],padding=True,truncation=True,
                      max_length=BENCHMARK_MAX_LENGTH,return_tensors='pt').to(device)
            with torch.inference_mode():_ = mdl(**batch).logits
        _sync()
    run(bench_texts[:min(BENCHMARK_BATCH,len(bench_texts))]);timings=[]
    for _ in range(BENCHMARK_REPEATS):
        _sync();start=time.perf_counter();run(bench_texts)
        timings.append(1000*(time.perf_counter()-start)/len(bench_texts))
    del mdl;gc.collect()
    if device.type=='cuda':torch.cuda.empty_cache()
    return {'task':'classification','model':name,'device':str(device),
            'inference_ms_per_sample':float(np.median(timings)),
            'benchmark_n':len(bench_texts),'batch_size':BENCHMARK_BATCH,'repeats':BENCHMARK_REPEATS}

latency=[]
for name in ['LogisticRegression','LinearSVM','MultinomialNB','RandomForest']:
    try:
        rec=bench_classical(name)
        if rec:latency.append(rec)
    except Exception as e:print('Benchmark failed:',name,repr(e))
for name in ['BanglaBERT','mBERT','MuRIL','XLM-R']:
    try:
        rec=bench_transformer(name)
        if rec:latency.append(rec)
    except Exception as e:print('Benchmark failed:',name,repr(e))
if latency:
    latency_df=pd.DataFrame(latency).drop_duplicates(['task','model'],keep='last')
    latency_df.to_csv(OUT/'metrics'/'classification_inference_latency.csv',index=False)
    display(latency_df)
else:print('No saved classifiers found for latency benchmarking.')


,task,model,device,inference_ms_per_sample,benchmark_n,batch_size,repeats
0,classification,LogisticRegression,CPU,0.090868,256,16,3
1,classification,LinearSVM,CPU,0.087571,256,16,3
2,classification,MultinomialNB,CPU,0.094628,256,16,3
3,classification,RandomForest,CPU,0.778065,256,16,3
4,classification,BanglaBERT,cuda,1.268198,256,16,3
5,classification,mBERT,cuda,1.720369,256,16,3
6,classification,MuRIL,cuda,1.145393,256,16,3
7,classification,XLM-R,cuda,1.865416,256,16,3


24. Final unified computational comparison and reproducibility audit
Merge measured inference latency without duplicating records; produce a single 300 DPI PNG per comparison. Existing figures with identical names are overwritten. No PDF exports.


Code cell 57

In [ ]:
import platform, sys, json, hashlib
METRICS=OUT/'metrics';FIGURES=OUT/'figures'
METRICS.mkdir(parents=True,exist_ok=True);FIGURES.mkdir(parents=True,exist_ok=True)
base_path=METRICS/'computational_complexity.csv'
lat_path=METRICS/'classification_inference_latency.csv'
if base_path.exists():
    comp=pd.read_csv(base_path).drop_duplicates(['task','model'],keep='last')
else:
    comp=pd.DataFrame(columns=['task','model','train_seconds','parameters','gpu_peak_mb','disk_size_mb'])
if lat_path.exists():
    lat=pd.read_csv(lat_path)
    cols=['task','model','inference_ms_per_sample','benchmark_n','batch_size','repeats']
    comp=comp.drop(columns=[c for c in cols[2:] if c in comp.columns],errors='ignore')
    comp=comp.merge(lat[cols],on=['task','model'],how='outer',validate='one_to_one')
comp=comp.drop_duplicates(['task','model'],keep='last').sort_values(['task','model'])
comp.to_csv(METRICS/'computational_complexity_final.csv',index=False)
if (METRICS/'all_results.csv').exists():
    metrics=pd.read_csv(METRICS/'all_results.csv').drop_duplicates(['task','model','dataset'],keep='last')
    internal=metrics.loc[metrics.dataset.eq('internal_test')].copy()
    comparison=internal.merge(comp,on=['task','model'],how='left',validate='many_to_one')
    comparison.to_csv(METRICS/'reviewer_performance_vs_compute.csv',index=False)
    for task,metric in [('classification','macro_f1'),('standardization','chrf')]:
        sub=comparison[(comparison.task==task)].copy()
        if metric not in sub or 'train_seconds' not in sub:continue
        sub=sub.dropna(subset=[metric,'train_seconds'])
        if sub.empty:continue
        fig,ax=plt.subplots(figsize=(8,5))
        ax.scatter(sub.train_seconds,sub[metric],s=65)
        for _,r in sub.iterrows():ax.annotate(str(r.model),(r.train_seconds,r[metric]),xytext=(4,4),textcoords='offset points',fontsize=8)
        ax.set(xlabel='Recorded training time (seconds; device-dependent)',ylabel=metric,
               title=f'{task.capitalize()}: performance versus recorded training time')
        fig.tight_layout();fig.savefig(FIGURES/f'{task}_performance_vs_training_time.png',dpi=300,bbox_inches='tight');plt.close(fig)
    display(comparison[[c for c in ['task','model','dataset','macro_f1','chrf','train_seconds','inference_ms_per_sample','parameters','gpu_peak_mb','disk_size_mb'] if c in comparison]])
    assert not metrics.duplicated(['task','model','dataset']).any()

meta={'python':sys.version,'torch':torch.__version__,'transformers':transformers.__version__,
      'platform':platform.platform(),'seed':SEED,'near_threshold':NEAR_THRESHOLD,
      'train_val_test_target':[TRAIN_FRAC,VAL_FRAC,TEST_FRAC],
      'cuda_available':torch.cuda.is_available(),
      'gpu_name':torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
      'figures_format':'PNG 300 DPI only',
      'note':'CPU and GPU timings are device-dependent, not theoretical FLOPs'}
(OUT/'audit'/'reviewer_environment_manifest.json').write_text(json.dumps(meta,indent=2,ensure_ascii=False),encoding='utf-8')
print('Reviewer files saved. PNG count:',len(list(FIGURES.glob('*.png'))))
print('Duplicate metric keys:',0 if not (METRICS/'all_results.csv').exists() else int(pd.read_csv(METRICS/'all_results.csv').duplicated(['task','model','dataset']).sum()))


,task,model,dataset,macro_f1,chrf,train_seconds,inference_ms_per_sample,parameters,gpu_peak_mb,disk_size_mb
0,classification,BanglaBERT,internal_test,0.916778,NaN,773.967530,1.268198,164400389.0,3163.865234,632.985594
1,classification,LinearSVM,internal_test,0.942409,NaN,2.033739,0.087571,NaN,NaN,6.249054
2,classification,LogisticRegression,internal_test,0.926753,NaN,18.635544,0.090868,NaN,NaN,6.249180
3,classification,MuRIL,internal_test,0.924178,NaN,988.616055,1.145393,237560069.0,4560.170898,915.378068
4,classification,MultinomialNB,internal_test,0.919953,NaN,1.474203,0.094628,NaN,NaN,9.304772
5,classification,RandomForest,internal_test,0.883046,NaN,96.867648,0.778065,NaN,NaN,234.717465
6,classification,XLM-R,internal_test,0.918600,NaN,1120.920732,1.865416,278047493.0,5330.135742,1081.823714
7,classification,mBERT,internal_test,0.924652,NaN,868.032071,1.720369,177857285.0,3420.829102,682.235657
8,standardization,BanglaT5,internal_test,NaN,60.295243,4761.343632,NaN,247577856.0,4896.683594,945.557880
9,standardization,LSTM,internal_test,NaN,23.113532,44.765726,NaN,8368898.0,NaN,31.928327


Reviewer files saved. PNG count: 104
Duplicate metric keys: 0


25. Drive file uniqueness verification
Every result has a stable filename and overwrites its previous version. Existing PDFs from older runs are not deleted. For external validation, retain the strict filtered subsets; do not re-evaluate unfiltered source files.


Code cell 59

In [ ]:
from collections import Counter
results_path=OUT/'metrics'/'all_results.csv'
if results_path.exists():
    r=pd.read_csv(results_path)
    duplicate_keys=r[r.duplicated(['task','model','dataset'],keep=False)]
    print('Duplicate metric rows:',len(duplicate_keys))
    assert duplicate_keys.empty
for directory in ['metrics','figures','predictions','logs']:
    paths=list((OUT/directory).glob('*'))
    print(f'{directory}: {len(paths)} entries; PNG: {sum(p.suffix.lower()==".png" for p in paths)}; PDF: {sum(p.suffix.lower()==".pdf" for p in paths)}')
print('Note: files already saved by previous notebook versions remain unless manually removed.')


Duplicate metric rows: 0
metrics: 108 entries; PNG: 0; PDF: 0
figures: 104 entries; PNG: 104; PDF: 0
predictions: 50 entries; PNG: 0; PDF: 0
logs: 12 entries; PNG: 0; PDF: 0
Note: files already saved by previous notebook versions remain unless manually removed.


In [ ]:
import sys
import platform
import torch
import transformers
import sklearn

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("scikit-learn:", sklearn.__version__)

print("\nPlatform:", platform.platform())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "GPU Memory:",
        round(
            torch.cuda.get_device_properties(0).total_memory / 1024**3,
            2
        ),
        "GB"
    )
    print("CUDA:", torch.version.cuda)
else:
    print("GPU: Not available")

Python: 3.13.16
PyTorch: 2.14.1+cu130
Transformers: 5.22.1
scikit-learn: 1.6.1

Platform: Linux-6.6.159+-x86_64-with-glibc2.39
GPU: Tesla T4
GPU Memory: 14.56 GB
CUDA: 13.0
